# Publication figures

Consolidated, publication-quality figures for the digital-bat paper. Every analysis loads the **17 `DEFAULT_SESSIONS` once** (`use_kinematics=False`; Krishna sessions never appear) and uses the single consensus joint model `joint_v2_kin0_L4_d64_ff64_lam0.61_pw0.84_identity_we1.9_wd2.8_wa0.67_n17.pth` throughout. Shared style/stat/plot helpers live in `plotting_helpers.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sys.path.append(os.path.expanduser('~/Documents/code/utils'))
sys.path.append(os.path.expanduser('~/Documents/code/digitalbat'))

from digitalbat.utility.utils import (
    set_random_state, bat_palette, parse_bat, corr,
    poisson_fde_population, poisson_fde_population_filtered,
)
import notebook_helpers as nh
import plotting_helpers as ph

ph.set_pub_style()
ph.FIG_DIR = Path('out/figures')

seed = 1
set_random_state(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Shared setup

Load the 17 `DEFAULT_SESSIONS` **once** (`use_kinematics=False`, `behavior_dim==3`, Krishna excluded) and instantiate the single consensus joint model used throughout the paper. Downstream figures reuse these objects; nothing below re-loads data or re-instantiates the model.

In [ ]:
from digitalbat.datasets.data import bat_data_loader
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config
from digitalbat.datasets.kinematics import make_joint_dataloaders

# 17 DEFAULT_SESSIONS, position-only (behavior_dim == 3).
session_names = list(DEFAULT_SESSION_NAMES)
data_config = make_data_config(session_names=session_names)
session_names = data_config['session_names']
bin_width = data_config['bin_width']

# Single load in the joint model's *training* universe. make_joint_dataloaders
# loads once (kinematics on) and returns the position-only view `dl_pos` plus the
# shared meta. We use this ONE universe for BOTH the encode and decode arms so
# every analysis shares one flight_path_index (no cross-universe remapping). The
# tradeoff vs the plain position-only loader is that a few tail flights with short
# IMU are dropped — negligible, and it matches how the model was trained.
dataloaders, _dl_kin, unit_num_dict, meta_info_dict = make_joint_dataloaders(data_config)
behavior_dim = 3

bats = ph.bat_order()
BAT_PALETTE = ph.bat_pal()
n_neurons_total = sum(unit_num_dict.values())
print(f'{len(session_names)} sessions | {n_neurons_total} neurons | behavior_dim={behavior_dim}')

In [ ]:
from digitalbat.models.joint import JointMaskedTransformer

# --- Consensus model: the single joint encode/decode transformer used paper-wide.
JOINT_CKPT = ('./out/joint/joint_v2_kin0_L4_d64_ff64_lam0.61_pw0.84_identity'
              '_we1.9_wd2.8_wa0.67_n17.pth')
JOINT_PARAMS = dict(num_layers=4, latent_dim=64, num_heads=4, ff_dim=64, dropout=0.2)
INPUT_TRANSFORM = 'identity'


def joint_session_room(s):
    """3-config room map (matches joint-encode-decode.ipynb `session_room`)."""
    if '_fs' in s:
        return 'krishna_fs'
    if s.startswith('Dataset_2_'):
        return 'dataset_2'
    return 'lab'


session_to_room = {s: joint_session_room(s) for s in unit_num_dict}

set_random_state(seed)
joint_model = JointMaskedTransformer(
    unit_num_dict, behavior_dim=3, input_transform=INPUT_TRANSFORM,
    session_to_room=session_to_room, **JOINT_PARAMS).to(device)
joint_model.load_state_dict(torch.load(JOINT_CKPT, map_location=device))
joint_model = joint_model.eval()

# Encode and decode arms share ONE flight universe (see the data-load cell), so the
# decode-arm loaders/meta are simply the shared ones — no separate universe.
joint_loaders, joint_meta = dataloaders, meta_info_dict
print('loaded consensus joint model:', JOINT_CKPT)

In [ ]:
# Encode-arm adapter + per-neuron encode-metric helper now live in the package
# (digitalbat.models.joint), keeping this notebook focused on the figures.
from digitalbat.models.joint import JointEncoderAdapter, encode_eval_records

bat_former = JointEncoderAdapter(joint_model, unit_num_dict).to(device).eval()

---
## Figure 1 — Transformer encoding/decoding models outperform standard approaches

**a–b** Model schematics (omitted — drawn separately). **c** Predicted vs observed rate maps for two dual-field example units. **d–e** Per-neuron encoding fit (Poisson FDE, Pearson r) for GLM (place), GLM (trajectory-specific place), and the joint transformer. **f** Example decoded 3-D trajectory (joint). **g–h** Decoding accuracy (3-D Fréchet distance per flight; 3-D reconstruction error per point) for NB (place), unified CEBRA, and joint.

In [ ]:
# --- Joint DECODE arm: predict 3D position from spikes (all behaviour masked).
# Per-timestep records feed the trajectory panel (1f), reconstruction error (1h),
# and per-flight Fréchet distance (1g). Uses the joint decode loaders + meta.
from digitalbat.decoding import discrete_frechet

TRUTH_3D = ['x_true', 'y_true', 'z_true']
PRED_3D = ['x_pred', 'y_pred', 'z_pred']


@torch.no_grad()
def joint_decode_records(split='test'):
    recs = []
    for sn, sd in joint_loaders[split].items():
        bm = np.asarray(joint_meta['behavior_mean'][sn]).reshape(-1)[:3]
        bs = np.asarray(joint_meta['behavior_std'][sn]).reshape(-1)[:3]
        bat = parse_bat(sn)
        for batch in sd:
            m = batch.mask.to(device)
            bt = batch.mask.clone().to(device)      # decode: all behaviour masked
            nt = torch.zeros_like(bt)
            _, beh_pred, _ = joint_model(sn, batch.spike.to(device),
                                         batch.behavior.to(device), m, nt, bt)
            yp = beh_pred[:, :, :3].cpu().numpy() * bs + bm
            yt = batch.behavior[:, :, :3].numpy() * bs + bm
            mn = batch.mask.numpy().astype(bool)
            B, T = mn.shape
            fpi = np.broadcast_to(batch.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf = yt[mn], yp[mn]
            for i in range(len(ytf)):
                recs.append(dict(bat=bat, session=sn, flight_id=int(fpi[mn][i]),
                                 t_idx=int(tix[mn][i]),
                                 x_true=ytf[i, 0], y_true=ytf[i, 1], z_true=ytf[i, 2],
                                 x_pred=ypf[i, 0], y_pred=ypf[i, 1], z_pred=ypf[i, 2],
                                 err_2d=float(np.linalg.norm(ytf[i, :2] - ypf[i, :2])),
                                 err_3d=float(np.linalg.norm(ytf[i] - ypf[i]))))
    return pd.DataFrame(recs).assign(decoder='joint')


def frechet_per_flight(df, truth=TRUTH_3D, pred=PRED_3D):
    """Per-(decoder, session, flight) discrete 3D Fréchet distance."""
    rows = []
    gcols = ['bat', 'session', 'flight_id']
    gcols = (['decoder'] + gcols) if 'decoder' in df else gcols
    for keys, grp in df.groupby(gcols, sort=False, observed=True):
        grp = grp.sort_values('t_idx')
        row = dict(zip(gcols, keys if isinstance(keys, tuple) else (keys,)))
        row['frechet_3d'] = discrete_frechet(grp[truth].to_numpy(float),
                                             grp[pred].to_numpy(float))
        rows.append(row)
    return pd.DataFrame(rows)


joint_decode_test = joint_decode_records('test')
joint_frechet = frechet_per_flight(joint_decode_test)
print('joint decode: %d timepoints, %d flights' % (len(joint_decode_test), len(joint_frechet)))

In [ ]:
# === Fig 1c: dual-field example units (spikes / empirical rate / predicted rate) ===
# The paper's reference "Dataset_3_26659_240404 unit 148" was a typo for bat 29959;
# unit 1 is pinned to Dataset_3_29959_240404 unit 148 (peak sep ~6.6 m), unit 2 is
# auto-picked from a different bat. Edit FIG1C_PICKS to override.
FIG1C_PICKS = [('Dataset_3_29959_240404', 148)]   # pin unit 1; unit 2 auto (different bat)

from notebook_helpers import (build_flight_records, select_dual_field_cells,
                              plot_dual_field_cell, loc_bin_width_rm)

dt = bin_width
dual_candidates = []
for sn, sd in dataloaders['test'].items():
    Xl, yl, pl, ml, fl = [], [], [], [], []
    with torch.no_grad():
        for batch in sd:
            X = batch.behavior.to(device); m = batch.mask.to(device)
            rate = bat_former(sn, X, m)
            Xl.append(X.cpu().numpy()); yl.append(batch.spike.numpy())
            pl.append(rate.cpu().numpy()); ml.append(m.cpu().numpy())
            fl.append(batch.flight_type_index.numpy())
    Xa, ya, pa = np.vstack(Xl), np.vstack(yl), np.vstack(pl)
    ma, fa = np.vstack(ml), np.hstack(fl)
    yflat, pflat = ya[ma == 1.0], pa[ma == 1.0]
    ncorr = corr(yflat, pflat, axis=0)
    nfde = poisson_fde_population(yflat, pflat)
    r_lim = meta_info_dict['r_lim'][sn]
    flights = build_flight_records(Xa, ya, pa, ma, fa,
                                   meta_info_dict['behavior_mean'][sn],
                                   meta_info_dict['behavior_std'][sn])
    # n_top large so a specifically-requested unit is captured, not just the top 2.
    for c in select_dual_field_cells(flights, ya.shape[2], dt, r_lim,
                                     loc_bin_width=loc_bin_width_rm, n_top=100):
        dual_candidates.append(dict(session=sn, bat=parse_bat(sn), flights=flights,
                                    r_lim=r_lim, corr=float(ncorr[c['neuron']]),
                                    fde=float(nfde[c['neuron']]), **c))

by_key = {(d['session'], d['neuron']): d for d in dual_candidates}
cand_df = pd.DataFrame([{k: d[k] for k in ('session', 'bat', 'neuron', 'sep', 'corr', 'fde')}
                        for d in dual_candidates]).sort_values('sep', ascending=False)
print('Top dual-field candidates:'); print(cand_df.head(12).to_string(index=False))

dual_candidates.sort(key=lambda d: d['sep'], reverse=True)
picks = []
for p in (FIG1C_PICKS or []):
    if tuple(p) in by_key:
        picks.append(by_key[tuple(p)])
    else:
        print(f'WARNING: {p} not a dual-field candidate (thresholds); skipping pin.')
if not picks:                                   # fall back: best-separated cell
    picks = [dual_candidates[0]]
if len(picks) < 2:                              # add best cell from a different bat
    used = {pk['bat'] for pk in picks}
    picks.append(next((d for d in dual_candidates if d['bat'] not in used), dual_candidates[1]))

for pk in picks:
    title = (f"{pk['session']} — unit {pk['neuron']} | corr {pk['corr']:.2f}, "
             f"FDE {pk['fde']:.2f} | peak sep {pk['sep']:.1f} m")
    plot_dual_field_cell(pk['flights'], pk['neuron'], pk['clusters'], dt, pk['r_lim'],
                         title=title, peaks=pk['peaks'])
    plt.show()

In [ ]:
# === Fig 1d/e (+ ED Fig 1a/b): per-neuron encoding fit for GLM place, GLM
# trajectory-specific place, and the joint transformer ===
# Train-only Fréchet single-linkage flight clustering (alpha=0.8) supplies the
# trajectory-cluster place fields for the "joint"-conditioned GLM (matches
# model_comparison.ipynb / decode-position-comparison.ipynb).
from collections import Counter
from scipy.interpolate import interp1d
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from digitalbat.utility.utils import frechet_distance
from digitalbat.models.glm import (GLM_3DCosineBasis, GLM_Coupled,
                                   make_log_raised_cosine_basis)

CLUSTER_ALPHA, CLUSTER_DS, MIN_CLUSTER_FLIGHTS = 0.8, 7, 3
CLUSTER_MAX_ALPHA, UNCLUSTERED, N_LAGS = 1.5, -1, 10


def _downsample_traj(pos_xyz, mask, ds=CLUSTER_DS):
    out = np.full((pos_xyz.shape[0], ds, 3), np.nan)
    for i in range(pos_xyz.shape[0]):
        L = int(mask[i].sum())
        if L < 2:
            continue
        traj = pos_xyz[i, mask[i]]
        new_idx = np.linspace(0, L - 1, ds)
        for d in range(3):
            out[i, :, d] = interp1d(np.arange(L), traj[:, d])(new_idx)
    return out


def compute_flight_clusters(loaders, meta, alpha=CLUSTER_ALPHA):
    flight_cluster, cluster_keys = {}, {}
    for sn in loaders['train']:
        bm = np.asarray(meta['behavior_mean'][sn]).reshape(-1)
        bs = np.asarray(meta['behavior_std'][sn]).reshape(-1)
        trajs, fpis = {}, {}
        for split in ('train', 'valid', 'test'):
            ds = loaders[split][sn].dataset
            pos = (ds.behavior.numpy() * bs + bm)[..., :3]
            trajs[split] = _downsample_traj(pos, ds.mask.numpy().astype(bool))
            fpis[split] = ds.flight_path_index.numpy().astype(int)
        Xtr = trajs['train']
        keep = np.where(~np.isnan(Xtr[:, 0, 0]))[0]
        X, n = Xtr[keep], len(keep)
        mapping = {}
        if n < 2:
            for split in ('train', 'valid', 'test'):
                for f in fpis[split]:
                    mapping[int(f)] = UNCLUSTERED
            flight_cluster[sn], cluster_keys[sn] = mapping, []
            continue
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                D[i, j] = D[j, i] = frechet_distance(X[i], X[j])
        used_alpha = alpha
        while True:
            Z = linkage(squareform(D, checks=False), method='single')
            labels = fcluster(Z, t=used_alpha, criterion='distance').astype(int)
            if len(np.unique(labels)) > 1 or used_alpha >= CLUSTER_MAX_ALPHA:
                break
            used_alpha += 0.1
        counts = Counter(labels)
        major = sorted(int(c) for c, k in counts.items() if k >= MIN_CLUSTER_FLIGHTS)
        majset = set(major)
        for p, gi in enumerate(keep):
            lab = int(labels[p])
            mapping[int(fpis['train'][gi])] = lab if lab in majset else UNCLUSTERED
        for gi in range(Xtr.shape[0]):
            mapping.setdefault(int(fpis['train'][gi]), UNCLUSTERED)
        for split in ('valid', 'test'):
            t = trajs[split]
            for i in range(t.shape[0]):
                f = int(fpis[split][i])
                if np.isnan(t[i, 0, 0]):
                    mapping[f] = UNCLUSTERED
                    continue
                dists = np.array([frechet_distance(t[i], X[j]) for j in range(n)])
                jm = int(np.argmin(dists))
                lab = int(labels[jm])
                mapping[f] = lab if (dists[jm] <= used_alpha and lab in majset) else UNCLUSTERED
        flight_cluster[sn], cluster_keys[sn] = mapping, major
    return flight_cluster, cluster_keys


def cluster_idx_from_fpi(fpi_array, session, flight_cluster):
    fmap = flight_cluster.get(session, {})
    return torch.tensor([fmap.get(int(f), UNCLUSTERED) for f in fpi_array], dtype=torch.long)


flight_cluster, cluster_keys_by_session = compute_flight_clusters(dataloaders, meta_info_dict)

# --- Build + load the two GLMs and evaluate per-neuron FDE/corr on test.
_glm_common = dict(n_bases_per_dim=16, pos_range=(-3.0, 3.0), width_to_spacing_ratio=4,
                   activation='exp', bin_width=bin_width)


@torch.no_grad()
def eval_glm_metrics(model, joint_cond=False):
    model = model.to(device).eval()
    rows = []
    is_coupled = isinstance(model, GLM_Coupled)
    for sn, sd in dataloaders['test'].items():
        yv, yp = [], []
        for b in sd:
            m = b.mask.to(device)
            ci = (cluster_idx_from_fpi(b.flight_path_index.numpy(), sn, flight_cluster).to(device)
                  if (is_coupled and joint_cond) else None)
            if is_coupled:
                out = model(sn, b.behavior.to(device), m, spike=b.spike.to(device), cluster_idx=ci)
            else:
                out = model(sn, b.behavior.to(device), m)
            v = m == 1.0
            yv.append(b.spike.to(device)[v].cpu().numpy()); yp.append(out[v].cpu().numpy())
        yv, yp = np.vstack(yv), np.vstack(yp)
        c, f = corr(yv, yp, axis=0), poisson_fde_population(yv, yp)
        for u in range(yv.shape[1]):
            rows.append(dict(session=sn, bat=parse_bat(sn), unit=u,
                             corr=float(c[u]), fde=float(f[u])))
    return pd.DataFrame(rows)


set_random_state(seed)
glm_place = GLM_3DCosineBasis(**_glm_common, output_dim_dict=unit_num_dict)
glm_place.load_state_dict(torch.load(
    './out/bat_glm/bat_glm_position_seed_1_optuna_acc50_default17.pth', map_location=device))
set_random_state(seed)
glm_traj = GLM_Coupled(output_dim_dict=unit_num_dict, **_glm_common, hist_basis=None,
                       coup_basis=None, acausal_basis=None, n_lags=N_LAGS,
                       cluster_keys=cluster_keys_by_session)
glm_traj.load_state_dict(torch.load(
    './out/bat_glm/bat_glm_joint_seed_1_optuna_acc50_default17.pth', map_location=device))

ENC_LABELS = {'glm_place': 'GLM (place)', 'glm_traj': 'GLM (traj. place)', 'joint': 'Joint'}
ENC_ORDER = ['glm_place', 'glm_traj', 'joint']
encode_metrics_df = pd.concat([
    eval_glm_metrics(glm_place, joint_cond=False).assign(model='glm_place'),
    eval_glm_metrics(glm_traj, joint_cond=True).assign(model='glm_traj'),
    encode_eval_records(bat_former, dataloaders, device).assign(model='joint'),
], ignore_index=True)
encode_metrics_df['model'] = pd.Categorical(encode_metrics_df['model'], ENC_ORDER, ordered=True)
print('encode metrics rows:', len(encode_metrics_df))

In [ ]:
# Fig 1d (FDE) and 1e (corr): session medians + per-neuron, stars vs Joint.
fig_d, ptab_d = ph.compare_models_fig(
    encode_metrics_df, 'fde', ENC_ORDER, ref='joint', labels=ENC_LABELS,
    ylabel='Poisson FDE', unit_name='neuron', clip=(-1, 1), zero_line=True)
fig_d.suptitle('Figure 1d — encoding fit (Poisson FDE)', y=1.03)
ph.save_fig(fig_d, 'fig1d_fde'); plt.show()

fig_e, ptab_e = ph.compare_models_fig(
    encode_metrics_df, 'corr', ENC_ORDER, ref='joint', labels=ENC_LABELS,
    ylabel='Pearson r', unit_name='neuron', clip=(-0.2, 1), zero_line=True)
fig_e.suptitle('Figure 1e — encoding fit (Pearson r)', y=1.03)
ph.save_fig(fig_e, 'fig1e_corr'); plt.show()

print('\nFig 1d — FDE, session-level Wilcoxon vs Joint:')
ph.p_table(ptab_d.to_dict('records'))
print('\nFig 1e — corr, session-level Wilcoxon vs Joint:')
ph.p_table(ptab_e.to_dict('records'))

In [ ]:
# === Fig 1f: example decoded 3D trajectory (joint decoder only) ===
from digitalbat.decoding import polylines_from_per_timestep_df, plot_trajectory_zoom_3d

# Pick an indoor session with many decoded flights for a clean zoom panel.
_counts = (joint_decode_test[joint_decode_test['session'].map(ph.is_indoor)]
           .groupby('session')['flight_id'].nunique())
zoom_session = _counts.idxmax() if len(_counts) else joint_decode_test['session'].iloc[0]

joint_polylines = polylines_from_per_timestep_df(
    joint_decode_test, pred_cols=('x_pred', 'y_pred', 'z_pred'),
    truth_cols=('x_true', 'y_true', 'z_true'), sort_by='t_idx')
fig_f = plot_trajectory_zoom_3d(
    zoom_session, [('joint', 'Joint (enc-dec)', 'tab:olive')],
    {'joint': joint_polylines}, figsize_per_panel=(5, 5),
    suptitle=f'Figure 1f — decoded vs true trajectories ({zoom_session})')
ph.save_fig(fig_f, 'fig1f_trajectory'); plt.show()

In [ ]:
# === Fig 1g/h baseline: naïve-Bayes (place) decoder ===
# Ported from decode-position-comparison.ipynb: per-session NB-2D/3D on the base
# dataloaders, hyperparameters from the cached Optuna studies.
from digitalbat.models.decoder import NaiveBayes_Decoder

nb_best = {}
for ndim in (2, 3):
    with open(f'./out/nb_decoder/nb_decoder_{ndim}d_seed_{seed}_optuna.pkl', 'rb') as fh:
        nb_best[ndim] = pickle.load(fh).best_params


def gather_session_arrays(loader, bm, bs):
    spk, beh, msk, fpi = [], [], [], []
    for batch in loader:
        spk.append(batch.spike.numpy()); beh.append(batch.behavior.numpy())
        msk.append(batch.mask.numpy().astype(bool)); fpi.append(batch.flight_path_index.numpy())
    spk = np.concatenate(spk); beh = np.concatenate(beh)
    msk = np.concatenate(msk); fpi = np.concatenate(fpi)
    bm_a, bs_a = np.asarray(bm).squeeze(), np.asarray(bs).squeeze()
    pos = beh[..., :3] * bs_a[None, None, :3] + bm_a[None, None, :3]
    B, T, _ = beh.shape
    fpit = np.broadcast_to(fpi[:, None], (B, T)); tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
    return dict(spikes=spk[msk], pos_x=pos[..., 0][msk], pos_y=pos[..., 1][msk],
                pos_z=pos[..., 2][msk], flight_id=fpit[msk].astype(int), t_idx=tix[msk].astype(int))


def make_nb(ndim, r_lim, p):
    n_bins = (p['n_bins_x'], p['n_bins_y'], p['n_bins_z']) if ndim == 3 else (p['n_bins_x'], p['n_bins_y'])
    return NaiveBayes_Decoder(n_bins=n_bins, lims=[tuple(r_lim[d]) for d in range(ndim)],
                              sigma=p['sigma'], dt=bin_width, min_occupancy=p['min_occupancy'],
                              use_uniform_prior=p['use_uniform_prior'])


def eval_nb_records():
    recs = []
    for sn in dataloaders['test']:
        bm, bs = meta_info_dict['behavior_mean'][sn], meta_info_dict['behavior_std'][sn]
        r_lim = meta_info_dict['r_lim'][sn]
        tr = gather_session_arrays(dataloaders['train'][sn], bm, bs)
        te = gather_session_arrays(dataloaders['test'][sn], bm, bs)
        if tr['spikes'].shape[0] < 100 or te['spikes'].shape[0] < 10:
            continue
        tr3 = np.column_stack([tr['pos_x'], tr['pos_y'], tr['pos_z']])
        te3 = np.column_stack([te['pos_x'], te['pos_y'], te['pos_z']])
        nb3 = make_nb(3, r_lim, nb_best[3]).fit(tr['spikes'], tr3)
        pd3, _ = nb3.predict(te['spikes'])
        err3 = np.sqrt(((te3 - pd3) ** 2).sum(1))
        for i in range(len(err3)):
            recs.append(dict(bat=parse_bat(sn), session=sn, flight_id=int(te['flight_id'][i]),
                             t_idx=int(te['t_idx'][i]), x_true=te3[i, 0], y_true=te3[i, 1],
                             z_true=te3[i, 2], x_pred=pd3[i, 0], y_pred=pd3[i, 1], z_pred=pd3[i, 2],
                             err_3d=float(err3[i])))
    return pd.DataFrame(recs)


nb_decode_test = eval_nb_records()
nb_frechet = frechet_per_flight(nb_decode_test)
print('NB (place) decode: %d points, %d flights' % (len(nb_decode_test), len(nb_frechet)))

In [ ]:
# === Fig 1g/h baseline: unified CEBRA decoder ===
# Ported from cebra-comparison.ipynb. Guarded: needs the source-only CEBRA repo on
# the path (~/Documents/code/CEBRA) and its cached unified solver; on any failure
# the panel simply omits CEBRA (a message is printed).
import hashlib
from sklearn.neighbors import KNeighborsRegressor

CEBRA_OK = True
try:
    sys.path.insert(0, os.path.expanduser('~/Documents/code/CEBRA'))
    import cebra, cebra.data, cebra.models, cebra.solver
    from cebra import CEBRA

    OUT_DIR, CEBRA_MAX_ITER, KNN_N, UNI_VARIANT = './out/cebra', 6000, 36, 'cebra_pos'
    CEBRA_HP = dict(model_architecture='offset10-model', learning_rate=3e-4, temperature=1.0,
                    time_offsets=10, output_dimension=3, num_hidden_units=32)
    UNIFIED_MASKS = {'RandomNeuronMask': (0.3, 0.9, 0.05), 'RandomTimestepMask': (0.3, 0.9, 0.05)}
    _sess_tag = hashlib.md5('|'.join(session_names).encode()).hexdigest()[:8]

    def _hp_tag():
        a = CEBRA_HP['model_architecture'].replace('-model', '')
        return (f"{a}_lr{CEBRA_HP['learning_rate']:.0e}_t{CEBRA_HP['temperature']:.2g}"
                f"_to{CEBRA_HP['time_offsets']}_d{CEBRA_HP['output_dimension']}_h{CEBRA_HP['num_hidden_units']}")

    def session_flights(session, split):
        bm = np.asarray(meta_info_dict['behavior_mean'][session]).squeeze()[:3]
        bs = np.asarray(meta_info_dict['behavior_std'][session]).squeeze()[:3]
        out = []
        for batch in dataloaders[split][session]:
            m = batch.mask.numpy().astype(bool)
            spike = batch.spike.numpy(); pos_z = batch.behavior[:, :, :3].numpy()
            fpi = batch.flight_path_index.numpy()
            for b in range(m.shape[0]):
                sel = m[b]
                if sel.sum() == 0:
                    continue
                out.append(dict(spike=spike[b, sel].astype(np.float32),
                                pos=(pos_z[b, sel] * bs[None] + bm[None]).astype(np.float32),
                                flight_id=int(fpi[b]), t_idx=np.flatnonzero(sel).astype(int)))
        return out

    def build_unified_solver(neuron_counts):
        model = cebra.models.init(CEBRA_HP['model_architecture'], num_neurons=int(sum(neuron_counts)),
                                  num_units=CEBRA_HP['num_hidden_units'],
                                  num_output=CEBRA_HP['output_dimension']).to(device)
        crit = cebra.models.FixedCosineInfoNCE(temperature=CEBRA_HP['temperature']).to(device)
        opt = torch.optim.Adam(list(model.parameters()), lr=CEBRA_HP['learning_rate'])
        return cebra.solver.UnifiedSolver(model=model, criterion=crit, optimizer=opt, tqdm_on=False)

    def fit_unified(Xs, Ys, path):
        solver = build_unified_solver([X.shape[1] for X in Xs])
        if os.path.exists(path):
            solver.load_state_dict(torch.load(path, map_location=device, weights_only=False), strict=True)
            return solver
        dsets = [cebra.data.TensorDataset(neural=Xs[i].astype(np.float32),
                                          continuous=Ys[i].astype(np.float32), device=device)
                 for i in range(len(Xs))]
        dataset = cebra.data.UnifiedDataset(dd for dd in dsets)
        dataset.set_masks(UNIFIED_MASKS); dataset.to(device)
        loader = cebra.data.UnifiedLoader(dataset, conditional='time_delta', num_steps=CEBRA_MAX_ITER,
                                          batch_size=512, time_offset=CEBRA_HP['time_offsets'])
        dataset.configure_for(solver.model)
        set_random_state(seed); solver.fit(loader)
        solver.save(logdir=os.path.dirname(path), filename=os.path.basename(path))
        return solver

    def unified_embed_split(solver, split):
        fb = {s: session_flights(s, split) for s in session_names}
        Xs_all = [np.vstack([f['spike'] for f in fb[s]]).astype(np.float32) for s in session_names]
        Ys_all = [np.vstack([f['pos'] for f in fb[s]]).astype(np.float32) for s in session_names]
        Xt, Yt = [torch.from_numpy(X) for X in Xs_all], [torch.from_numpy(Y) for Y in Ys_all]
        solver = solver.to('cpu'); out = {}
        for i, s in enumerate(session_names):
            bs_ = min(512, max(2, len(Xs_all[i]) // 3))
            emb = solver.transform(Xt, Yt, session_id=i, batch_size=bs_).cpu().numpy()
            pieces, c = [], 0
            for f in fb[s]:
                L = len(f['spike']); pieces.append((emb[c:c + L], f)); c += L
            out[s] = pieces
        solver.to(device)
        return out

    _train_by = {s: session_flights(s, 'train') for s in session_names}
    Xs = [np.vstack([f['spike'] for f in _train_by[s]]).astype(np.float32) for s in session_names]
    Ys = [np.vstack([f['pos'] for f in _train_by[s]]).astype(np.float32) for s in session_names]
    unified_path = f'{OUT_DIR}/unified_{UNI_VARIANT}__{_hp_tag()}__it{CEBRA_MAX_ITER}__seed{seed}__s{len(session_names)}_{_sess_tag}.pt'
    unified_model = fit_unified(Xs, Ys, unified_path)

    uni_tr, uni_te = unified_embed_split(unified_model, 'train'), unified_embed_split(unified_model, 'test')
    crecs = []
    for s in session_names:
        Etr = np.vstack([e for e, _ in uni_tr[s]]); Ytr = np.vstack([f['pos'] for _, f in uni_tr[s]])
        knn = KNeighborsRegressor(n_neighbors=KNN_N, metric='cosine').fit(Etr, Ytr)
        for emb, f in uni_te[s]:
            pred, true = np.asarray(knn.predict(emb)), f['pos']
            e3 = np.sqrt(((true - pred) ** 2).sum(1))
            for i in range(len(true)):
                crecs.append(dict(bat=parse_bat(s), session=s, flight_id=f['flight_id'],
                                  t_idx=int(f['t_idx'][i]), x_true=true[i, 0], y_true=true[i, 1],
                                  z_true=true[i, 2], x_pred=pred[i, 0], y_pred=pred[i, 1],
                                  z_pred=pred[i, 2], err_3d=float(e3[i])))
    cebra_decode_test = pd.DataFrame(crecs)
    cebra_frechet = frechet_per_flight(cebra_decode_test)
    print('unified CEBRA decode: %d points, %d flights' % (len(cebra_decode_test), len(cebra_frechet)))
except Exception as exc:
    CEBRA_OK = False
    cebra_decode_test = cebra_frechet = None
    print(f'[skip CEBRA] {type(exc).__name__}: {exc}')

In [ ]:
# === Fig 1g/h: decoding accuracy for NB (place), unified CEBRA, joint ===
DECODE_LABELS = {'naive_bayes': 'NB (place)', 'cebra': 'unified CEBRA', 'joint': 'Joint'}
DECODE_ORDER = ['naive_bayes', 'cebra', 'joint']

# Assemble per-flight Fréchet (1g) and per-point recon error (1h) across decoders.
_fr = [joint_frechet.assign(decoder='joint'), nb_frechet.assign(decoder='naive_bayes')]
_rc = [joint_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='joint'),
       nb_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='naive_bayes')]
if cebra_frechet is not None:
    _fr.append(cebra_frechet.assign(decoder='cebra'))
    _rc.append(cebra_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='cebra'))

decode_frechet_df = pd.concat(_fr, ignore_index=True)
decode_recon_df = pd.concat(_rc, ignore_index=True)
present = [d for d in DECODE_ORDER if d in set(decode_frechet_df['decoder'])]
for df in (decode_frechet_df, decode_recon_df):
    df['decoder'] = pd.Categorical(df['decoder'], present, ordered=True)

fig_g, ptab_g = ph.compare_models_fig(
    decode_frechet_df, 'frechet_3d', present, ref='joint', labels=DECODE_LABELS,
    model_col='decoder', ylabel='3D Fréchet (m)', unit_name='flight', higher_is_better=False)
fig_g.suptitle('Figure 1g — decoding: 3D Fréchet distance', y=1.03)
ph.save_fig(fig_g, 'fig1g_frechet'); plt.show()

fig_h, ptab_h = ph.compare_models_fig(
    decode_recon_df, 'err_3d', present, ref='joint', labels=DECODE_LABELS,
    model_col='decoder', ylabel='3D recon error (m)', unit_name='point', higher_is_better=False)
fig_h.suptitle('Figure 1h — decoding: 3D reconstruction error', y=1.03)
ph.save_fig(fig_h, 'fig1h_recon'); plt.show()

print('\nFig 1g — 3D Fréchet, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_g.to_dict('records'))
print('\nFig 1h — 3D recon error, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_h.to_dict('records'))

**Figure 1 | Transformer-based encoding/decoding models outperform standard approaches on hippocampal data.**
**a–b**, Model schematics (masking scheme; dual-branch architecture), drawn separately.
**c**, Two example units with spatially separated place fields on two different flight routes. For each unit and route: observed spikes (red, area ∝ count) on the flight path, the empirical firing-rate map, and the joint model's predicted-rate map (empirical and predicted share a colour scale).
**d–e**, Encoding fit quantified as per-neuron Poisson fraction deviance explained (**d**) and Pearson correlation (**e**) between predicted and observed spike trains on held-out flights, for a place GLM, a trajectory-specific-place GLM, and the joint transformer. Left, session medians (points = sessions, coloured by bat); right, all neurons pooled. Boxes show median and IQR; whiskers 1.5× IQR.
**f**, Representative decoded (olive) versus true (black) 3-D flight trajectories from the joint decoder for one session.
**g–h**, Decoding accuracy as per-flight 3-D discrete Fréchet distance (**g**) and per-point 3-D reconstruction error (**h**) for a naïve-Bayes place decoder, unified CEBRA, and the joint decoder.
Stars denote two-sided Wilcoxon signed-rank tests on paired per-session medians versus the joint model (*P* < 0.05 *, *P* < 0.01 **, *P* < 0.001 ***); n.s. comparisons are unmarked. Exact *P* values are printed in the tables above each panel. FDE axes are clipped to [−1, 1] for display; all statistics use raw values.

---
## Extended Data Figure 1 — full model ladders

**a–b** Per-neuron encoding fit (Poisson FDE, Pearson r) for every model in `model_comparison.ipynb`. **c–d** Per-flight/point decoding accuracy (3-D Fréchet distance, 3-D reconstruction error) for the full decoder ladder in `decode-position-comparison.ipynb`. Stars: session-level Wilcoxon vs the joint model. (`GLM + lags + kin` is omitted — it requires the kinematics-augmented behaviour, incompatible with the position-only single load; re-enable with a separate kinematics loader if needed.)

In [ ]:
# === ED Fig 1a/b: all encoding models (per-neuron FDE + corr) ===
from digitalbat.models.former import Bat_Transformer
from digitalbat.models.former_defaults import (make_model_config, make_model_name,
                                              DEFAULT_LR, DEFAULT_WEIGHT_DECAY)
from digitalbat.models.glm import make_symmetric_raised_cosine_basis

# Temporal-filter bases (match model_comparison.ipynb).
HIST_N_BASIS, COUP_N_BASIS, ACAUSAL_N_BASIS = 6, 4, 6
hist_basis, _, _ = make_log_raised_cosine_basis(HIST_N_BASIS, N_LAGS, bin_width)
coup_basis, _, _ = make_log_raised_cosine_basis(COUP_N_BASIS, N_LAGS, bin_width, include_zero_lag=True)
acausal_basis, _, _ = make_symmetric_raised_cosine_basis(ACAUSAL_N_BASIS, N_LAGS, bin_width)
glm_lags = np.arange(-1.0, 1.0 + 1e-9, 0.2).tolist()
DATA_TAG, GLM_TAG = '_acc50_preln_17_sess', '_acc50_default17'


def build_glm(name, cond):
    ck = cluster_keys_by_session if cond == 'joint' else None
    if name == 'GLM' and cond == 'position':
        return GLM_3DCosineBasis(**_glm_common, output_dim_dict=unit_num_dict)
    if name == 'GLM + lags':
        return GLM_3DCosineBasis(**_glm_common, output_dim_dict=unit_num_dict, lag_seconds=glm_lags)
    return GLM_Coupled(output_dim_dict=unit_num_dict, **_glm_common,
                       hist_basis=hist_basis if 'hist' in name else None,
                       coup_basis=coup_basis if 'coup' in name else None,
                       acausal_basis=acausal_basis if 'acausal' in name else None,
                       n_lags=N_LAGS, cluster_keys=ck)


def glm_ckpt(name, cond):
    tag = name.lower().replace(' + ', '_').replace(' ', '_')
    return f'./out/bat_glm/bat_{tag}_{cond}_seed_{seed}_optuna{GLM_TAG}.pth'


# Full-arch transformer config from the pre-LN Optuna search.
import json as _json
_ep = f'./out/bat_former/former_best_params{DATA_TAG}_seed_{seed}.json'
_arch = ({k: _json.load(open(_ep))[k] for k in ('latent_dim', 'ff_dim', 'num_layers', 'dropout_prob')
          if k in _json.load(open(_ep))} if os.path.exists(_ep) else {})
base_cfg = make_model_config(behavior_dim, unit_num_dict, meta_info_dict, **_arch)


def build_former(kind):
    if kind == 'transformer':
        return Bat_Transformer(**base_cfg)
    if kind == 'mlp':
        return Bat_Transformer(**{**base_cfg, 'use_attention': False})
    return Bat_Transformer(**{**base_cfg, 'latent_dim': 32, 'ff_dim': 32, 'num_layers': 1})


@torch.no_grad()
def eval_former_metrics(model):
    model = model.to(device).eval()
    rows = []
    for sn, sd in dataloaders['test'].items():
        yv, yp = [], []
        for b in sd:
            m = b.mask.to(device)
            out = model(sn, b.behavior.to(device), m)
            v = m == 1.0
            yv.append(b.spike.to(device)[v].cpu().numpy()); yp.append(out[v].cpu().numpy())
        yv, yp = np.vstack(yv), np.vstack(yp)
        c, f = corr(yv, yp, axis=0), poisson_fde_population(yv, yp)
        for u in range(yv.shape[1]):
            rows.append(dict(session=sn, bat=parse_bat(sn), unit=u, corr=float(c[u]), fde=float(f[u])))
    return pd.DataFrame(rows)


# Registry: (display label, kind, spec). Joint GLMs get a '(joint)' suffix.
ED1_MODELS = [
    ('GLM (place)',        'glm',    ('GLM', 'position')),
    ('GLM + lags',         'glm',    ('GLM + lags', 'position')),
    ('GLM + hist',         'glm',    ('GLM + hist', 'position')),
    ('GLM + coup',         'glm',    ('GLM + coup', 'position')),
    ('GLM + acausal',      'glm',    ('GLM + acausal', 'position')),
    ('GLM (traj.)',        'glm',    ('GLM', 'joint')),
    ('GLM + hist (traj.)', 'glm',    ('GLM + hist', 'joint')),
    ('GLM + coup (traj.)', 'glm',    ('GLM + coup', 'joint')),
    ('GLM + acausal (traj.)', 'glm', ('GLM + acausal', 'joint')),
    ('MLP only',           'former', 'mlp'),
    ('encode-small',       'former', 'small'),
    ('Transformer',        'former', 'transformer'),
    ('Joint',              'joint',  None),
]
ED1_ORDER = [m[0] for m in ED1_MODELS]

_pieces = []
for label, kind, spec in ED1_MODELS:
    try:
        if kind == 'glm':
            name, cond = spec
            set_random_state(seed)
            gm = build_glm(name, cond)
            gm.load_state_dict(torch.load(glm_ckpt(name, cond), map_location=device))
            df = eval_glm_metrics(gm, joint_cond=(cond == 'joint'))
        elif kind == 'former':
            set_random_state(seed)
            fm = build_former(spec)
            ck = make_model_name(seed, DEFAULT_LR, DEFAULT_WEIGHT_DECAY,
                                 out_dir={'transformer': './out/bat_former', 'mlp': './out/bat_former_no_attn',
                                          'small': './out/bat_former_small'}[spec], suffix=DATA_TAG)
            fm.load_state_dict(torch.load(ck, map_location=device))
            df = eval_former_metrics(fm)
        else:
            df = encode_eval_records(bat_former, dataloaders, device)
        _pieces.append(df.assign(model=label))
        print(f'  ok: {label}')
    except Exception as exc:
        print(f'  [skip {label}] {type(exc).__name__}: {exc}')

ed1_encode_df = pd.concat(_pieces, ignore_index=True)
ed1_present = [m for m in ED1_ORDER if m in set(ed1_encode_df['model'])]
ed1_encode_df['model'] = pd.Categorical(ed1_encode_df['model'], ed1_present, ordered=True)
print('ED1 encode models:', ed1_present)

In [ ]:
# ED Fig 1a (FDE) and 1b (corr): all encoding models, stars vs Joint.
fig_ed1a, ptab_ed1a = ph.compare_models_fig(
    ed1_encode_df, 'fde', ed1_present, ref='Joint', ylabel='Poisson FDE',
    unit_name='neuron', clip=(-1, 1), zero_line=True, figsize=(1.15 * len(ed1_present) + 3, 5))
fig_ed1a.suptitle('ED Fig 1a — encoding fit (FDE), all models', y=1.02)
ph.save_fig(fig_ed1a, 'edfig1a_fde'); plt.show()

fig_ed1b, ptab_ed1b = ph.compare_models_fig(
    ed1_encode_df, 'corr', ed1_present, ref='Joint', ylabel='Pearson r',
    unit_name='neuron', clip=(-0.2, 1), zero_line=True, figsize=(1.15 * len(ed1_present) + 3, 5))
fig_ed1b.suptitle('ED Fig 1b — encoding fit (corr), all models', y=1.02)
ph.save_fig(fig_ed1b, 'edfig1b_corr'); plt.show()

print('\nED Fig 1a — FDE, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_ed1a.to_dict('records'))
print('\nED Fig 1b — corr, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_ed1b.to_dict('records'))

In [ ]:
# === ED Fig 1c/d: decoder ladder (extends Fig 1g/h with transformer + NB+continuity) ===
# Ladder: NB (place), NB + continuity, Transformer decoder, unified CEBRA, joint.
# The SVM and GLM-inversion decoders from decode-position-comparison.ipynb are the
# most custom (bespoke classes); they are deferred here — add analogously if needed.
from digitalbat.models.decoder import NaiveBayes_Continuity_Decoder, Bat_Transformer_Decoder

ladder_fr = [joint_frechet.assign(decoder='joint'), nb_frechet.assign(decoder='naive_bayes')]
ladder_rc = [joint_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='joint'),
             nb_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='naive_bayes')]
if cebra_frechet is not None:
    ladder_fr.append(cebra_frechet.assign(decoder='cebra'))
    ladder_rc.append(cebra_decode_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='cebra'))


# --- NB + continuity (cached c_speed) ---
def eval_nb_cont_records():
    with open(f'./out/nb_decoder/nb_cont_decoder_3d_seed_{seed}_optuna.pkl', 'rb') as fh:
        c_speed = pickle.load(fh).best_params['c_speed']
    recs = []
    for sn in dataloaders['test']:
        bm, bs = meta_info_dict['behavior_mean'][sn], meta_info_dict['behavior_std'][sn]
        r_lim = meta_info_dict['r_lim'][sn]
        tr, te = (gather_session_arrays(dataloaders[s][sn], bm, bs) for s in ('train', 'test'))
        if tr['spikes'].shape[0] < 100 or te['spikes'].shape[0] < 10:
            continue
        tr3 = np.column_stack([tr['pos_x'], tr['pos_y'], tr['pos_z']])
        te3 = np.column_stack([te['pos_x'], te['pos_y'], te['pos_z']])
        p = nb_best[3]
        nbc = NaiveBayes_Continuity_Decoder(
            c_speed=c_speed, sigma_min=0.02, sigma_max=5.0,
            n_bins=(p['n_bins_x'], p['n_bins_y'], p['n_bins_z']),
            lims=[tuple(r_lim[d]) for d in range(3)], sigma=p['sigma'], dt=bin_width,
            min_occupancy=p['min_occupancy'], use_uniform_prior=p['use_uniform_prior']).fit(
            tr['spikes'], tr3, flight_id=tr['flight_id'], t_idx=tr['t_idx'], dt=bin_width)
        pd3, _ = nbc.predict(te['spikes'], flight_id=te['flight_id'], t_idx=te['t_idx'])
        err3 = np.sqrt(((te3 - pd3) ** 2).sum(1))
        for i in range(len(err3)):
            recs.append(dict(bat=parse_bat(sn), session=sn, flight_id=int(te['flight_id'][i]),
                             t_idx=int(te['t_idx'][i]), x_true=te3[i, 0], y_true=te3[i, 1],
                             z_true=te3[i, 2], x_pred=pd3[i, 0], y_pred=pd3[i, 1],
                             z_pred=pd3[i, 2], err_3d=float(err3[i])))
    return pd.DataFrame(recs)


try:
    nbc_test = eval_nb_cont_records()
    ladder_fr.append(frechet_per_flight(nbc_test).assign(decoder='nb_continuity'))
    ladder_rc.append(nbc_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='nb_continuity'))
    print('  ok: NB + continuity')
except Exception as exc:
    print(f'  [skip NB+continuity] {type(exc).__name__}: {exc}')


# --- Transformer decoder (cached best checkpoint) ---
@torch.no_grad()
def eval_transformer_decoder():
    tdec = Bat_Transformer_Decoder(
        latent_dim=128, ff_dim=32, num_layers=1, dropout_prob=0.30, readin_dropout=0.19,
        input_transform='identity', input_dim_dict=unit_num_dict, output_dim=3,
        num_heads=4, final_activation=nn.Identity()).to(device)
    ckpt = ('./out/bat_decoder/bat_decoder_seed_1_latent_dim_128_ff_dim_32_num_layers_1'
            '_dropout_prob_0.30_readin_dropout_0.19_input_transform_identity.pth')
    tdec.load_state_dict(torch.load(ckpt, map_location=device))
    tdec.eval()
    recs = []
    for sn, sd in dataloaders['test'].items():
        bm = meta_info_dict['behavior_mean'][sn].squeeze(); bs = meta_info_dict['behavior_std'][sn].squeeze()
        for b in sd:
            m = b.mask.to(device)
            yp = tdec(sn, b.spike.to(device), m).cpu().numpy() * bs[None, None, :3] + bm[None, None, :3]
            yt = b.behavior[:, :, :3].numpy() * bs[None, None, :3] + bm[None, None, :3]
            mn = b.mask.numpy().astype(bool); B, T = mn.shape
            fpi = np.broadcast_to(b.flight_path_index.numpy()[:, None], (B, T))
            tix = np.broadcast_to(np.arange(T)[None, :], (B, T))
            ytf, ypf = yt[mn], yp[mn]
            for i in range(len(ytf)):
                recs.append(dict(bat=parse_bat(sn), session=sn, flight_id=int(fpi[mn][i]),
                                 t_idx=int(tix[mn][i]), x_true=ytf[i, 0], y_true=ytf[i, 1],
                                 z_true=ytf[i, 2], x_pred=ypf[i, 0], y_pred=ypf[i, 1],
                                 z_pred=ypf[i, 2], err_3d=float(np.linalg.norm(ytf[i] - ypf[i]))))
    return pd.DataFrame(recs)


try:
    tdec_test = eval_transformer_decoder()
    ladder_fr.append(frechet_per_flight(tdec_test).assign(decoder='transformer'))
    ladder_rc.append(tdec_test[['bat', 'session', 'flight_id', 'err_3d']].assign(decoder='transformer'))
    print('  ok: Transformer decoder')
except Exception as exc:
    print(f'  [skip Transformer decoder] {type(exc).__name__}: {exc}')

ED1_DEC_LABELS = {'naive_bayes': 'NB (place)', 'nb_continuity': 'NB + continuity',
                  'transformer': 'Transformer', 'cebra': 'unified CEBRA', 'joint': 'Joint'}
ED1_DEC_ORDER = ['naive_bayes', 'nb_continuity', 'transformer', 'cebra', 'joint']
ed1_frechet = pd.concat(ladder_fr, ignore_index=True)
ed1_recon = pd.concat(ladder_rc, ignore_index=True)
ed1_dec_present = [d for d in ED1_DEC_ORDER if d in set(ed1_frechet['decoder'])]
for df in (ed1_frechet, ed1_recon):
    df['decoder'] = pd.Categorical(df['decoder'], ed1_dec_present, ordered=True)

fig_ed1c, ptab_ed1c = ph.compare_models_fig(
    ed1_frechet, 'frechet_3d', ed1_dec_present, ref='joint', labels=ED1_DEC_LABELS,
    model_col='decoder', ylabel='3D Fréchet (m)', unit_name='flight', higher_is_better=False)
fig_ed1c.suptitle('ED Fig 1c — decoding: 3D Fréchet, full ladder', y=1.02)
ph.save_fig(fig_ed1c, 'edfig1c_frechet'); plt.show()

fig_ed1d, ptab_ed1d = ph.compare_models_fig(
    ed1_recon, 'err_3d', ed1_dec_present, ref='joint', labels=ED1_DEC_LABELS,
    model_col='decoder', ylabel='3D recon error (m)', unit_name='point', higher_is_better=False)
fig_ed1d.suptitle('ED Fig 1d — decoding: 3D recon error, full ladder', y=1.02)
ph.save_fig(fig_ed1d, 'edfig1d_recon'); plt.show()

print('\nED Fig 1c — 3D Fréchet, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_ed1c.to_dict('records'))
print('\nED Fig 1d — 3D recon error, session-level Wilcoxon vs Joint:'); ph.p_table(ptab_ed1d.to_dict('records'))

**Extended Data Figure 1 | Full model ladders.** **a–b**, Per-neuron encoding fit (Poisson FDE, Pearson r) on held-out flights for the complete model set: place and temporal-filter GLMs (history, coupling, acausal self-filter) under position and trajectory-cluster conditioning, an attention-free MLP, a small encoder-only transformer, the full transformer, and the joint model. **c–d**, Per-flight 3-D Fréchet distance and per-point 3-D reconstruction error for the decoder ladder (naïve-Bayes place, NB + continuity, transformer decoder, unified CEBRA, joint). Points are session medians coloured by bat (left panels) and pooled neurons/flights (right panels); stars are two-sided Wilcoxon signed-rank tests on paired per-session medians versus the joint model; exact *P* values are printed above. FDE axes clipped to [−1, 1] for display; statistics use raw values. `GLM + lags + kin` (needs kinematics) and the SVM / GLM-inversion decoders (bespoke classes) are omitted here.

---
## Extended Data Figure 2 — synthetic & shuffle controls

**a–b** Encoding fit on synthetic place-cell spikes (same rate maps, held-out). **c–f** Decoding cost (metric − real) by 3-D Fréchet for: model ladder vs joint, trial-shuffle, synthetic place-only, synthetic place×trajectory. **g–j** As c–f for 3-D reconstruction error.

> **Scope note.** These control panels are not a load-and-evaluate of cached artifacts: each mode (`trial_shuffled`, `synthetic`, `synthetic_traj`) requires **re-training** decoders on surrogate/shuffled training data (via `digitalbat.datasets.synthetic.make_synthetic_train_dataloaders[_traj]` and the shuffle builders), then evaluating on real test data — the full per-mode pipeline in `decode-position-comparison.ipynb`. To keep `figures.ipynb` a cached-artifact plotting notebook, ED2 is built by running that pipeline in the source notebook and importing its per-mode `frechet_med` / `err_med` tables here. Wire the import below once those tables are exported. The `cost = metric(mode) − metric(real)` per-session-median plot + Wilcoxon then reuses `ph.compare_models_fig` / `ph.paired_session_test` exactly as elsewhere.

---
## Figure 2 — Transformer latents warp physical space to emphasise route identity

Physical XYZ, the 64-d transformer latent (PCA→3D), and the unified-CEBRA latent, each coloured by **x-coordinate** (top row) and by **flight route** (bottom row). Restricted to the indoor lab room (Datasets 1 & 3). The latent-extraction cell below is reused by Figure 4.

In [ ]:
# --- Shared latent extraction (encode pass): per-timepoint neural latent h_n,
# predicted rate, true position, and per-flight route label. Reused by Fig 4.
from notebook_helpers import DIST_THRESH, get_target_locs


@torch.no_grad()
def extract_latents(split='test'):
    """Concatenate encode-pass latents + metadata across all sessions.

    Returns a dict with arrays aligned per valid timepoint: `latent` (N, d),
    `pred` (N, K_session — ragged, so returned per-session too), `xyz` (N, 3),
    `session`, `bat`, `flight_id`, `rel_t`, and per-point `route`/`origin`/`dest`.
    """
    lat, xyz, sess, batv, fid, relt = [], [], [], [], [], []
    per_session_latent = {}
    for sn in session_names:
        bm = meta_info_dict['behavior_mean'][sn][0, 0, :3]
        bs = meta_info_dict['behavior_std'][sn][0, 0, :3]
        sl = []
        for batch in dataloaders[split][sn]:
            X = batch.behavior.to(device)
            m = batch.mask.to(device)
            B, T, _ = X.shape
            spike0 = X.new_zeros(B, T, unit_num_dict[sn])
            _, _, xf = joint_model(sn, spike0, X, m, m, torch.zeros_like(m))
            h_n = xf[:, :T].cpu().numpy()
            Xn = batch.behavior.numpy()
            mn = batch.mask.numpy() == 1.0
            fpi = batch.flight_path_index.numpy()
            for b in range(B):
                v = mn[b]
                nv = int(v.sum())
                if nv == 0:
                    continue
                lat.append(h_n[b, v]); sl.append(h_n[b, v])
                pos = Xn[b, :, :3] * bs + bm
                xyz.append(pos[v])
                sess.append(np.full(nv, sn)); batv.append(np.full(nv, parse_bat(sn)))
                fid.append(np.full(nv, int(fpi[b])))
                relt.append(np.arange(nv) / max(nv - 1, 1))
        per_session_latent[sn] = np.concatenate(sl) if sl else np.empty((0, joint_model.latent_dim))
    out = dict(latent=np.concatenate(lat), xyz=np.vstack(xyz),
               session=np.concatenate(sess), bat=np.concatenate(batv),
               flight_id=np.concatenate(fid), rel_t=np.concatenate(relt),
               per_session_latent=per_session_latent)

    # Per-(session, flight) origin/destination endpoint classes -> route id.
    origin = np.zeros(len(out['session']), int)
    dest = np.zeros_like(origin)
    key = np.char.add(np.char.add(out['session'], '|'), out['flight_id'].astype(str))
    for k in np.unique(key):
        m = key == k
        sn = out['session'][m][0]
        rew, perch = get_target_locs(sn)
        order = np.argsort(out['rel_t'][m])
        pos = out['xyz'][m][order]
        o = ph.endpoint_class(pos[0], rew, perch, DIST_THRESH)
        d = ph.endpoint_class(pos[-1], rew, perch, DIST_THRESH)
        origin[m] = o; dest[m] = d
    out['origin'] = origin
    out['dest'] = dest
    out['route'] = np.array([ph.route_id(o, d) for o, d in zip(origin, dest)])
    return out


emb = extract_latents('test')
print('latent points:', len(emb['session']), '| latent dim:', emb['latent'].shape[1])

In [ ]:
# === Figure 2 a-f: physical / transformer-latent / CEBRA embeddings (indoor only) ===
# Rows: colour by x-coordinate (a-c) and by flight route (d-f).
rng = np.random.default_rng(0)
ind = np.where(np.array([ph.is_indoor(s) for s in emb['session']]))[0]
if len(ind) > 40000:
    ind = np.sort(rng.choice(ind, 40000, replace=False))
xyz_i, sess_i, route_i = emb['xyz'][ind], emb['session'][ind], emb['route'][ind]
colx_i = ph.zscore_by_session(xyz_i[:, 0], sess_i)

# Physical (top-down x-y) and transformer latent (64d -> lobe-rotated 2D).
lat2d, _lat_tf = ph.lobe_projection(emb['latent'][ind], route_i)
panels = [('Physical (x–y)', xyz_i[:, :2], colx_i, route_i),
          ('Transformer latent', lat2d, colx_i, route_i)]

# Unified CEBRA embedding for the same indoor test flights (guarded on CEBRA).
if cebra_frechet is not None:
    ce, cx, croute = [], [], []
    for s in session_names:
        if not ph.is_indoor(s):
            continue
        rew, perch = get_target_locs(s)
        for embp, f in uni_te[s]:
            pos = f['pos']
            o = ph.endpoint_class(pos[0], rew, perch, DIST_THRESH)
            d = ph.endpoint_class(pos[-1], rew, perch, DIST_THRESH)
            ce.append(embp); cx.append(pos[:, 0])
            croute.append(np.full(len(embp), ph.route_id(o, d)))
    ce = np.vstack(ce); croute = np.concatenate(croute)
    cx = ph.zscore_by_session(np.concatenate(cx),
                              np.concatenate([np.full(len(e), s) for s in session_names
                                              if ph.is_indoor(s) for e, _ in uni_te[s]]))
    cebra2d, _ = ph.lobe_projection(ce, croute)
    panels.append(('Unified CEBRA', cebra2d, cx, croute))

fig2, axes = plt.subplots(2, len(panels), figsize=(4.2 * len(panels), 8), squeeze=False)
xvmin, xvmax = np.percentile(colx_i, [2, 98])
for j, (title, e2, cxj, rj) in enumerate(panels):
    sc = ph.plot_embedding(axes[0, j], e2, cxj, cmap='viridis', vlim=(xvmin, xvmax), s=3)
    axes[0, j].set_title(title)
    ph.plot_embedding(axes[1, j], e2, rj, categorical=True, s=3)
axes[0, 0].set_ylabel('coloured by x')
axes[1, 0].set_ylabel('coloured by route')
fig2.colorbar(sc, ax=axes[0, :].tolist(), fraction=0.02, pad=0.01, label='x (z-scored)')
handles = [plt.Line2D([], [], marker='o', ls='', color=ph.ROUTE_COLORS[k], label=ph.ROUTE_LABELS[k])
           for k in range(len(ph.ROUTE_LABELS))]
axes[1, -1].legend(handles=handles, title='route', loc='center left',
                   bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
fig2.suptitle('Figure 2 — physical vs latent vs CEBRA embeddings (indoor sessions)', y=1.01)
ph.save_fig(fig2, 'fig2_embeddings'); plt.show()

**Figure 2 | The transformer latent embedding warps physical space to emphasise flight-route identity and align across animals.** Each point is one held-out flight time-bin from the indoor lab sessions (Datasets 1 & 3). Columns: true 3-D position (top-down x–y), the 64-dimensional transformer latent projected to 2-D (PCA followed by a rotation that separates the two dominant route lobes horizontally), and the unified-CEBRA latent (same rotation procedure). **a–c**, coloured by (per-session z-scored) x-coordinate. **d–f**, coloured by flight route (origin→destination endpoint class). Unlike physical space and the CEBRA embedding, the transformer latent segregates flights primarily by route identity rather than by instantaneous position.

---
## Figure 4 — Readout weights surface correlations in spatial and non-spatial activity

**a** PC1 of the readout-weight correlation vs mean firing rate, per session. **b** Residual-readout hierarchical clustering (dendrogram, similarity matrix, residual-readout matrix). **c–f, h, i** Mantel comparisons of the residual readout against non-flight rest, sleep, rate-map and coupling similarity. **g** Residual readouts predict CA1 vs CA3 membership (SVM). **j–k** Attribution matrices (example; future−past by cluster).

> Panels **c–f, h, i** build the rate-map, non-flight-rest and coupling similarity streams directly (rate maps from flight spikes; non-flight from a per-session `Data_Prep` load; coupling from the GLM+history+coupling checkpoint). The **sleep** matrix (5th panel of **c**) matches flight units to Dataset_0 sleep units via UnitMatch `MatchTable.csv` files (guarded — skipped if those / the Dataset_0 spike files are absent).

In [ ]:
# === Figure 4 foundation: per-neuron readout weights + metadata + subregion ===
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram

# Per-neuron table: unit-normed readout rows, pType, probe, position, mean FR.
_rows, _W = [], []
for sn in dataloaders['test']:
    W = bat_former.readouts[sn][0].weight.detach().cpu().numpy()   # (K, D)
    pos = np.asarray(meta_info_dict['neuron_position'][sn])
    ptype = np.asarray(meta_info_dict['pType'][sn])
    probe = np.asarray(meta_info_dict['probe_id'][sn])
    # mean FR from test spikes
    yl, ml = [], []
    for b in dataloaders['test'][sn]:
        yl.append(b.spike.numpy()); ml.append(b.mask.numpy())
    ya, ma = np.vstack(yl), np.vstack(ml)
    mfr = ya[ma == 1.0].mean(0) / bin_width
    for u in range(W.shape[0]):
        pt = ptype[u] if not isinstance(ptype[u], np.ndarray) else 'filtered'
        _rows.append(dict(session=sn, bat=parse_bat(sn), unit=u, pType=str(pt),
                          probe_id=int(probe[u]), lateral=float(pos[u, 0]),
                          depth=float(pos[u, 1]), mean_fr=float(mfr[u])))
    _W.append(W)
readout_df = pd.DataFrame(_rows)
W_all = np.vstack(_W)
readout_df['readout_weight'] = list(W_all / np.linalg.norm(W_all, axis=1, keepdims=True))

# Attach joint per-neuron corr/FDE (computed for Fig 1d/e).
_jm = encode_metrics_df[encode_metrics_df['model'] == 'joint'][['session', 'unit', 'corr', 'fde']]
readout_df = readout_df.merge(_jm.rename(columns={'corr': 'corr_all', 'fde': 'fde_all'}),
                              on=['session', 'unit'], how='left')


# --- Subregion (CA1/CA3) from per-(bat,probe) depth clumps (internals recipe) ---
CLUMP_HW, PEAK_DIST, HBIN, SM_SIGMA, PROM = 300, 450, 25, 200, 0.04


def detect_clumps(depths):
    if len(depths) < 5:
        return np.array([])
    lo = np.floor(depths.min() / HBIN) * HBIN
    hi = np.ceil(depths.max() / HBIN) * HBIN + HBIN
    counts, edges = np.histogram(depths, bins=np.arange(lo, hi, HBIN))
    centers = 0.5 * (edges[:-1] + edges[1:])
    sm = gaussian_filter1d(counts.astype(float), sigma=SM_SIGMA / HBIN)
    if sm.max() == 0:
        return np.array([])
    padded = np.concatenate([[0.0], sm, [0.0]])
    pk, _ = find_peaks(padded, distance=max(1, PEAK_DIST // HBIN), prominence=PROM * sm.max())
    pk = pk - 1
    pk = pk[(pk >= 0) & (pk < len(sm))]
    if len(pk) == 0:
        pk = np.array([int(np.argmax(sm))])
    if len(pk) > 2:
        pk = pk[np.argsort(sm[pk])[::-1][:2]]
    return np.sort(centers[pk])


def _window(peak, lo, hi):
    a, b = peak - CLUMP_HW, peak + CLUMP_HW
    if b > hi:
        b, a = hi, hi - 2 * CLUMP_HW
    elif a < lo:
        a, b = lo, lo + 2 * CLUMP_HW
    return a, b


readout_df['subregion'] = None
for (bat, probe), grp in readout_df.groupby(['bat', 'probe_id']):
    d = grp['depth'].values
    peaks = detect_clumps(d)
    if len(d) == 0 or len(peaks) == 0:
        continue
    dlo, dhi = float(np.nanmin(d)), float(np.nanmax(d))
    labels = np.array([None] * len(d), dtype=object)
    if len(peaks) == 1:
        a, b = _window(peaks[0], dlo, dhi)
        labels[(d >= a) & (d <= b)] = 'CA1'
    else:
        ventral, dorsal = peaks[0], peaks[-1]
        la, lb = _window(dorsal, dlo, dhi)
        va, vb = _window(ventral, dlo, dhi)
        if vb > la:
            vb, va = la, la - 2 * CLUMP_HW
        labels[(d >= la) & (d <= lb)] = 'CA1'
        labels[(d >= va) & (d <= vb)] = 'CA3'
    readout_df.loc[grp.index, 'subregion'] = labels


def residual_readout(W, fr):
    """Project centered mean-FR axis out of unit-normed rows W, renormalise."""
    ax = W.T @ (fr - fr.mean())
    n = np.linalg.norm(ax)
    Wr = W - np.outer(W @ (ax / n), ax / n) if n > 0 else W.copy()
    rn = np.linalg.norm(Wr, axis=1, keepdims=True); rn[rn == 0] = 1.0
    return Wr / rn


# Well-fit cells (corr_all > 0.2), residual readout per session.
clu_df = readout_df[readout_df['corr_all'] > 0.2].copy().reset_index(drop=True)
clu_df['readout_weight_residual'] = clu_df['readout_weight']
for _, grp in clu_df.groupby('session'):
    Wr = residual_readout(np.stack(grp['readout_weight'].values), grp['mean_fr'].values)
    for j, ix in zip(range(len(grp)), grp.index):
        clu_df.at[ix, 'readout_weight_residual'] = Wr[j]
print('readout_df:', len(readout_df), 'neurons |', readout_df['subregion'].notna().sum(),
      'with subregion | clu_df well-fit:', len(clu_df))

In [ ]:
# === Fig 4a: PC1 of readout weights correlates with mean firing rate ===
from sklearn.decomposition import PCA as _PCA

_rec = []
for sn, grp in readout_df.groupby('session'):
    W = np.stack(grp['readout_weight'].values)
    if len(grp) < 4:
        continue
    pc1 = _PCA(n_components=1).fit_transform(W).ravel()
    r = np.corrcoef(pc1, grp['mean_fr'].values)[0, 1]
    # sign is arbitrary; align so positive = PC1 tracks FR
    _rec.append(dict(session=sn, bat=parse_bat(sn), r=abs(r)))
pc1_fr_df = pd.DataFrame(_rec)

_res = ph.onesample_session_test(pc1_fr_df, 'r', session_col='session', popmean=0.0)
fig4a, ax = plt.subplots(figsize=(4, ph.PANEL_H + 1))
sns.stripplot(data=pc1_fr_df, y='r', hue='bat', palette=BAT_PALETTE,
              hue_order=[b for b in bats if b in set(pc1_fr_df['bat'])],
              size=9, jitter=0.15, edgecolor='k', linewidth=0.4, ax=ax, legend=False)
ax.axhline(0, color='0.5', ls='--', lw=1)
ax.set(xlabel='', ylabel='|corr(PC1, mean FR)|', xticks=[])
ax.set_title('Fig 4a')
# star vs 0 (one-sample) drawn as a corner annotation
ph.sig_bracket(ax, -0.3, 0.3, _res['p'], base=0.96)
ph.bat_legend(ax, loc='lower right')
ph.save_fig(fig4a, 'fig4a_pc1_fr'); plt.show()
print('Fig 4a — |corr(PC1, mean FR)| vs 0 (one-sample Wilcoxon):')
ph.p_table([dict(comparison='PC1-FR corr vs 0', n=_res['n'], median=_res['med'], p=_res['p'])])

In [ ]:
# === Fig 4b: residual-readout hierarchical clustering (dendrogram + matrices) ===
from scipy.cluster.hierarchy import fcluster, set_link_color_palette
import matplotlib.colors as mcolors

Wr = np.stack(clu_df['readout_weight_residual'].values)     # (N, D), unit-normed
dist = pdist(Wr, metric='cosine')
Z = linkage(dist, method='average')

N_CLUSTERS = 5
clu_df['cluster'] = fcluster(Z, t=N_CLUSTERS, criterion='maxclust')
CLUSTER_ORDER = sorted(clu_df['cluster'].unique())
CLUSTER_PALETTE = {c: col for c, col in zip(
    CLUSTER_ORDER, sns.color_palette('Set2', len(CLUSTER_ORDER)))}

dend = dendrogram(Z, no_plot=True)
order = dend['leaves']
sim_ord = (Wr @ Wr.T)[np.ix_(order, order)]
Wr_ord = Wr[order]
clus_ord = clu_df['cluster'].values[order]

fig4b = plt.figure(figsize=(16, 6))
gs = fig4b.add_gridspec(1, 3, width_ratios=[1.1, 1.4, 1.4], wspace=0.25)

ax_d = fig4b.add_subplot(gs[0])
set_link_color_palette([mcolors.to_hex(CLUSTER_PALETTE[c]) for c in CLUSTER_ORDER])
dendrogram(Z, ax=ax_d, color_threshold=Z[-(N_CLUSTERS - 1), 2], no_labels=True,
           above_threshold_color='0.6')
ax_d.set(title='Fig 4b — dendrogram', xlabel='neurons', ylabel='cosine distance')
set_link_color_palette(None)

ax_s = fig4b.add_subplot(gs[1])
im = ax_s.imshow(sim_ord, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto', interpolation='nearest')
ax_s.set(title='residual-readout cosine similarity', xlabel='neuron', ylabel='neuron')
fig4b.colorbar(im, ax=ax_s, fraction=0.046, pad=0.04)
# cluster colour strip
strip = np.array([CLUSTER_PALETTE[c] for c in clus_ord])
ax_s.imshow(strip[None, :, :], extent=(-0.5, len(order) - 0.5, len(order) + 0.5, len(order) + 6.5),
            aspect='auto')
ax_s.set_ylim(len(order) + 6.5, -0.5)

ax_w = fig4b.add_subplot(gs[2])
vlim = np.percentile(np.abs(Wr_ord), 98)
imw = ax_w.imshow(Wr_ord, cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto', interpolation='nearest')
ax_w.set(title='residual readout weights', xlabel='latent dim', ylabel='neuron (clustered)')
fig4b.colorbar(imw, ax=ax_w, fraction=0.046, pad=0.04)
ph.save_fig(fig4b, 'fig4b_clustering'); plt.show()
print('cluster sizes:', clu_df['cluster'].value_counts().sort_index().to_dict())

In [ ]:
# === Fig 4c-f/h/i data streams: rate maps, non-flight rest corr, coupling ===
# Ported from internals.ipynb (rate maps, non-flight Data_Prep load, coupling GLM).
from digitalbat.datasets.data import Data_Prep_for_Raw
from notebook_helpers import loc_bin_width_rm

# (1) Per-neuron 2D rate maps (observed spikes / occupancy) per session.
session_rate_maps, session_occ = {}, {}
for sn, sd in dataloaders['test'].items():
    Xl, yl, ml = [], [], []
    for b in sd:
        Xl.append(b.behavior.numpy()); yl.append(b.spike.numpy()); ml.append(b.mask.numpy())
    Xa, ya, ma = np.vstack(Xl), np.vstack(yl), np.vstack(ml)
    bm, bs = meta_info_dict['behavior_mean'][sn], meta_info_dict['behavior_std'][sn]
    xlim, ylim = meta_info_dict['r_lim'][sn][0], meta_info_dict['r_lim'][sn][1]
    xe = np.arange(xlim[0], xlim[1] + loc_bin_width_rm, loc_bin_width_rm)
    ye = np.arange(ylim[0], ylim[1] + loc_bin_width_rm, loc_bin_width_rm)
    v = ma == 1.0
    lx = Xa[:, :, 0][v] * bs[0, 0, 0] + bm[0, 0, 0]
    ly = Xa[:, :, 1][v] * bs[0, 0, 1] + bm[0, 0, 1]
    spk = ya[v]
    occ, _, _ = np.histogram2d(lx, ly, bins=[xe, ye])
    under = occ < 3
    maps = np.full((spk.shape[1], (len(xe) - 1) * (len(ye) - 1)), np.nan)
    for u in range(spk.shape[1]):
        sm, _, _ = np.histogram2d(lx, ly, bins=[xe, ye], weights=spk[:, u])
        with np.errstate(divide='ignore', invalid='ignore'):
            rm = sm / (occ * bin_width)
        rm[under] = np.nan
        maps[u] = rm.ravel()
    session_rate_maps[sn] = maps
    session_occ[sn] = occ.ravel()

# (2) Non-flight rest correlation matrices (sqrt counts) per session.
nonflight_corr = {}
for sn in session_names:
    folder = data_config['parent_folder'] + sn
    su = data_config['parent_folder'] + '/To_transfer/out_Gunits_probe*_' + sn + '_with_metrics.mat'
    kw = dict(parent_folder=folder, su_file_path=su, use_speed_head_dir=False,
              use_IMU_flag=False, use_click_flag=False, use_LFP_flag=False,
              use_ripple_flag=False, spike_count_flag=True, bin_width=bin_width)
    try:
        dpf = Data_Prep_for_Raw(only_flight_flag=True, **kw)
        dpf.get_all_behavior(); dpf.get_all_spike()
        dpa = Data_Prep_for_Raw(only_flight_flag=False, **kw)
        dpa.get_all_behavior(); allspk = dpa.get_all_spike()
        nf = np.setdiff1d(np.arange(allspk.shape[1]), dpf.flight_ind)
        nonflight_corr[sn] = np.corrcoef(np.sqrt(allspk[:, nf]))
    except Exception as exc:
        print(f'  [nonflight skip {sn}] {type(exc).__name__}: {exc}')

# (3) Pairwise coupling similarity from the GLM+hist+coupling checkpoint.
session_coupling_sim = {}
COUP_CKPT = './out/bat_glm/bat_glm_hist_coup_position_seed_1_optuna_acc50_full_v2.pth'
if os.path.exists(COUP_CKPT):
    cs = torch.load(COUP_CKPT, map_location='cpu', weights_only=False)
    kint = cs['coup_basis'].numpy().sum(0) * bin_width
    for k, v in cs.items():
        if k.startswith('coup_weights.'):
            K = v.numpy() @ kint
            np.fill_diagonal(K, 0.0)
            Csym = 0.5 * (K + K.T); np.fill_diagonal(Csym, 0.0)
            session_coupling_sim[k.split('coup_weights.', 1)[1]] = Csym
print(f'streams: {len(session_rate_maps)} rate-map, {len(nonflight_corr)} non-flight, '
      f'{len(session_coupling_sim)} coupling sessions')


def ratemap_corr_matrix(maps):
    """Pairwise Pearson r between neurons' rate maps over jointly-valid bins."""
    n = maps.shape[0]
    C = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(i, n):
            m = np.isfinite(maps[i]) & np.isfinite(maps[j])
            if m.sum() >= 5 and maps[i, m].std() > 0 and maps[j, m].std() > 0:
                C[i, j] = C[j, i] = np.corrcoef(maps[i, m], maps[j, m])[0, 1]
    return C


session_sleep_corr = {}   # populated by the sleep-matching cell below


def session_matrices(sn):
    """Well-fit-neuron submatrices (nonflight, residual readout, rate map, coupling)
    for one session, all ordered by the session's clu_df units. Missing streams -> None."""
    grp = clu_df[clu_df['session'] == sn]
    units = grp['unit'].to_numpy()
    if len(units) < 6 or sn not in nonflight_corr:
        return None
    Wr = np.stack(grp['readout_weight_residual'].values)
    out = dict(units=units, grp=grp,
               nonflight=nonflight_corr[sn][np.ix_(units, units)],
               residual=Wr @ Wr.T,
               ratemap=ratemap_corr_matrix(session_rate_maps[sn][units]),
               coupling=(session_coupling_sim[sn][np.ix_(units, units)]
                         if sn in session_coupling_sim else None),
               sleep=session_sleep_corr.get(sn))
    return out


In [ ]:
# === Fig 4c sleep matrix: match flight units to Dataset_0 sleep units (UnitMatch) ===
# Ported from internals.ipynb. For each flight session with a sleep pair, map its
# Kilosort cluster ids to the sleep-session (Dataset_0) cluster ids via UnitMatch
# MatchTables, bin the sleep spikes, and build a per-session sleep FR-correlation
# matrix over the well-fit (clu_df) neurons (NaN where unmatched / silent). Guarded:
# sessions without MatchTable / Dataset_0 SU files are skipped.
from pathlib import Path

UM_ROOT = Path('/mnt/s5adam/Angelo/NP_datasets/UnitMatch')
SLEEP_PAIRS = {
    'Dataset_3_00000_240403': 'Dataset_0_00000_240403',
    'Dataset_3_00000_240404': 'Dataset_0_00000_240404',
    'Dataset_3_14543_240420': 'Dataset_0_14543_240420',
    'Dataset_3_29959_240403': 'Dataset_0_29959_240403',
    'Dataset_3_29959_240404': 'Dataset_0_29959_240404',
}


def _load_match_table(animal, date, probe):
    p = UM_ROOT / 'matches' / f'{animal}_{date}' / f'probe{probe}' / 'MatchTable.csv'
    return pd.read_csv(p) if p.is_file() else None


def _build_flight_to_sleep_map(animal, date):
    mapping = {}
    for probe in (1, 2, 3):
        df = _load_match_table(animal, date, probe)
        if df is None:
            continue
        cross = df[(df['RecSes 1'] != df['RecSes 2']) & (df['Matches'] == 1) & (df['RecSes 1'] == 1)]
        best = cross.sort_values('UM Probabilities', ascending=False).drop_duplicates('ID2', keep='first')
        for _, r in best.iterrows():
            mapping[(probe, int(r['ID2']))] = int(r['ID1'])
    return mapping


def _flight_cluster_ids(sn):
    dp = Data_Prep_for_Raw(
        parent_folder=data_config['parent_folder'] + '/' + sn,
        su_file_path=data_config['parent_folder'] + '/To_transfer/out_Gunits_probe*_' + sn + '_with_metrics.mat',
        use_speed_head_dir=False, use_IMU_flag=False, use_click_flag=False,
        use_LFP_flag=False, use_ripple_flag=False, spike_count_flag=True,
        bin_width=bin_width, only_flight_flag=True)
    dp.get_all_behavior(); dp.get_all_spike()
    return dp.cluster_id, dp.probe_id + 1   # data.py 0-indexed; UnitMatch uses probe 1..3


def _bin_ds0_spikes(ds0_session, sleep_clu_per_probe):
    binned = {}
    for probe in (1, 2, 3):
        wanted = sleep_clu_per_probe.get(probe)
        if not wanted:
            continue
        path = f"{data_config['parent_folder']}/To_transfer/out_Gunits_probe{probe}_{ds0_session}.mat"
        if not Path(path).is_file():
            continue
        ou = loadmat(path)['out_Gunits']
        all_cids = np.array([int(x[0][0, 0]) for x in ou['cluster_id']])
        all_times_s = [x[0][:, 0] / 1e6 for x in ou['spikeTimes_usec']]
        t_end = max((t.max() for t in all_times_s if len(t)), default=0.0) + bin_width
        edges = np.arange(0, t_end + bin_width, bin_width)
        out = np.zeros((len(wanted), len(edges) - 1), dtype=np.int32)
        for i, cid in enumerate(wanted):
            idx = np.where(all_cids == cid)[0]
            if len(idx):
                out[i] = np.histogram(all_times_s[idx[0]], bins=edges)[0]
        binned[probe] = out
    if not binned:
        return {}
    common = min(a.shape[1] for a in binned.values())
    return {p: a[:, :common] for p, a in binned.items()}


for sn, ds0 in SLEEP_PAIRS.items():
    grp = clu_df[clu_df['session'] == sn]
    if len(grp) < 3:
        continue
    animal, date = sn.split('_')[2], sn.split('_')[3]
    try:
        cid_full, pid_full = _flight_cluster_ids(sn)
        units = grp['unit'].to_numpy()
        if units.max() >= len(cid_full):
            continue
        fcid, fpid = cid_full[units], pid_full[units]
        fts = _build_flight_to_sleep_map(animal, date)
        if not fts:
            print(f'  [sleep skip {sn}] no MatchTable'); continue
        matched = [fts.get((int(fpid[i]), int(fcid[i]))) for i in range(len(units))]
        pos = [i for i, m in enumerate(matched) if m is not None]
        if len(pos) < 3:
            print(f'  [sleep {sn}] only {len(pos)} matched'); continue
        per_probe = {p: [] for p in (1, 2, 3)}
        for i in pos:
            per_probe[int(fpid[i])].append(int(matched[i]))
        binned = _bin_ds0_spikes(ds0, {p: l for p, l in per_probe.items() if l})
        if not binned:
            print(f'  [sleep skip {sn}] no Dataset_0 SU mats'); continue
        rows, cur = [], {p: 0 for p in (1, 2, 3)}
        for i in pos:
            p = int(fpid[i]); rows.append(binned[p][cur[p]]); cur[p] += 1
        sleep_mat = np.vstack(rows)
        C = np.full((len(units), len(units)), np.nan)
        var = sleep_mat.var(1); nz = var > 0
        idx_pos = np.array(pos)[nz]
        if len(idx_pos) >= 2:
            C[np.ix_(idx_pos, idx_pos)] = np.corrcoef(np.sqrt(sleep_mat[nz]))
        session_sleep_corr[sn] = C
        print(f'  sleep {sn}: {len(pos)} matched, {sleep_mat.shape[1] * bin_width / 60:.1f} min')
    except Exception as exc:
        print(f'  [sleep skip {sn}] {type(exc).__name__}: {exc}')
print(f'sleep matrices for {len(session_sleep_corr)} sessions')


In [ ]:
# === Fig 4c: example session — 4 similarity matrices sorted by non-flight rest ===
# (Sleep, the 5th matrix, needs the cross-probe sleep-cluster matching in
# internals.ipynb cell 59 and is added there; the other four are self-contained.)
_cand = [s for s in nonflight_corr if session_matrices(s) is not None]
_with_sleep = [s for s in _cand if session_matrices(s)['sleep'] is not None]
ex_session = max(_with_sleep or _cand, key=lambda s: len(clu_df[clu_df['session'] == s]))
M = session_matrices(ex_session)
# order by hierarchical clustering of the non-flight-rest matrix
nf = np.nan_to_num(M['nonflight'], nan=0.0)
d = 1 - np.abs(nf); np.fill_diagonal(d, 0.0)
Zex = linkage(squareform(d, checks=False), method='average')
order = dendrogram(Zex, no_plot=True)['leaves']
mats = [('non-flight rest', M['nonflight']), ('residual readout', M['residual']),
        ('rate map', M['ratemap']), ('coupling', M['coupling']), ('sleep', M['sleep'])]
fig4c, axes = plt.subplots(1, len(mats), figsize=(4.2 * len(mats), 4.4))
for ax, (name, mat) in zip(axes, mats):
    if mat is None:
        ax.axis('off'); ax.set_title(f'{name}\n(unavailable)'); continue
    Mo = mat[np.ix_(order, order)]
    vlim = np.nanpercentile(np.abs(Mo), 98)
    im = ax.imshow(Mo, cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto', interpolation='nearest')
    ax.set(title=name, xlabel='neuron', ylabel='neuron')
    fig4c.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig4c.suptitle(f'Fig 4c — {ex_session} (sorted by non-flight rest)', y=1.02)
ph.save_fig(fig4c, 'fig4c_matrices'); plt.show()


In [ ]:
# === Fig 4d: Mantel (non-flight rest vs residual readout / rate map / coupling);
#     4e/4f: partial Mantel controlling for rate map / coupling ===
N_MANTEL = 1000
rng_m = np.random.default_rng(42)
iu_cache = {}


def _triu(m):
    n = m.shape[0]; iu, ju = np.triu_indices(n, 1); return m[iu, ju]


mantel_rows, partial_rows = [], []
for sn in nonflight_corr:
    M = session_matrices(sn)
    if M is None:
        continue
    nf_vec = _triu(M['nonflight'])
    preds = {'residual readout': M['residual'], 'rate map': M['ratemap'], 'coupling': M['coupling']}
    for name, pmat in preds.items():
        if pmat is None:
            continue
        r, p, perm = nh.mantel(pmat, nf_vec, rng_m, N_MANTEL)
        mantel_rows.append(dict(comparison=name, session=sn, bat=parse_bat(sn),
                                r=r, p=p, perm_rs=perm))
    # partial Mantels: (non-flight, residual) | ratemap and the reverse; likewise coupling.
    res_vec, rm_vec = _triu(M['residual']), _triu(M['ratemap'])
    for ctrl_name, ctrl_mat in [('rate map', M['ratemap']), ('coupling', M['coupling'])]:
        if ctrl_mat is None:
            continue
        c_vec = _triu(ctrl_mat)
        r1, p1, pm1 = nh.partial_mantel(M['residual'], nf_vec, c_vec, rng_m, N_MANTEL)
        r2, p2, pm2 = nh.partial_mantel(ctrl_mat, nf_vec, res_vec, rng_m, N_MANTEL)
        partial_rows.append(dict(comparison=f'residual | {ctrl_name}', session=sn,
                                 bat=parse_bat(sn), r=r1, p=p1, perm_rs=pm1))
        partial_rows.append(dict(comparison=f'{ctrl_name} | residual', session=sn,
                                 bat=parse_bat(sn), r=r2, p=p2, perm_rs=pm2))

mantel_df = pd.DataFrame(mantel_rows)
mantel_df['comparison'] = pd.Categorical(mantel_df['comparison'],
                                         ['residual readout', 'rate map', 'coupling'], ordered=True)
fig4d = nh.plot_mantel_boxplots(mantel_df, ylabel='Mantel $\\rho$ (vs non-flight rest)', ncols=3)
fig4d.suptitle('Fig 4d — Mantel: non-flight rest vs readout / rate map / coupling', y=1.02)
ph.save_fig(fig4d, 'fig4d_mantel'); plt.show()

partial_df = pd.DataFrame(partial_rows)
for panel, ctrl in [('Fig 4e', 'rate map'), ('Fig 4f', 'coupling')]:
    sub = partial_df[partial_df['comparison'].str.contains(ctrl)].copy()
    if sub.empty:
        continue
    figp = nh.plot_mantel_boxplots(sub, ylabel='partial Mantel $\\rho$', ncols=2)
    figp.suptitle(f'{panel} — partial Mantel controlling for {ctrl}', y=1.02)
    ph.save_fig(figp, f'fig4{"e" if ctrl == "rate map" else "f"}_partial'); plt.show()

print('Fig 4d — Mantel (non-flight rest, per session):')
ph.p_table([dict(comparison=c, n=int((mantel_df['comparison'] == c).sum()),
                 median_r=float(mantel_df.loc[mantel_df['comparison'] == c, 'r'].median()),
                 frac_sig=float((mantel_df.loc[mantel_df['comparison'] == c, 'p'] < 0.05).mean()))
             for c in mantel_df['comparison'].cat.categories])


In [ ]:
# === Fig 4h: Mantel by subregion (CA1 vs CA3); 4i: spatial info by cluster ===
# 4h: within each subregion, Mantel of non-flight rest vs residual / rate map / coupling.
sub_rows = []
for sn in nonflight_corr:
    grp = clu_df[clu_df['session'] == sn]
    for sr in ('CA1', 'CA3'):
        g = grp[grp['subregion'] == sr]
        if len(g) < 6:
            continue
        units = g['unit'].to_numpy()
        nf = nonflight_corr[sn][np.ix_(units, units)]
        iu, ju = np.triu_indices(len(units), 1)
        Wr = np.stack(g['readout_weight_residual'].values)
        preds = {'residual readout': Wr @ Wr.T,
                 'rate map': ratemap_corr_matrix(session_rate_maps[sn][units]),
                 'coupling': (session_coupling_sim[sn][np.ix_(units, units)]
                              if sn in session_coupling_sim else None)}
        for name, pmat in preds.items():
            if pmat is None:
                continue
            r, p, _ = nh.mantel(pmat, nf[iu, ju], np.random.default_rng(1), 500)
            sub_rows.append(dict(comparison=name, subregion=sr, session=sn,
                                 bat=parse_bat(sn), r=r))
sub_mantel = pd.DataFrame(sub_rows)
fig4h, ax = plt.subplots(figsize=(7, ph.PANEL_H + 1))
order_h = ['residual readout', 'rate map', 'coupling']
sns.boxplot(data=sub_mantel, x='comparison', y='r', hue='subregion', order=order_h,
            hue_order=['CA1', 'CA3'], palette=nh.SUBREGION_COLORS, showfliers=False, ax=ax)
ax.axhline(0, color='0.5', ls='--', lw=1)
ax.set(xlabel='', ylabel='Mantel $\\rho$ (vs non-flight)', title='Fig 4h — by subregion')
ph.save_fig(fig4h, 'fig4h_subregion'); plt.show()

# 4i: spatial information (bits/spike) per neuron, by readout cluster.
def spatial_info(rmap, occ):
    p = occ / occ.sum()
    valid = np.isfinite(rmap) & (occ > 0)
    r = rmap[valid]; pv = p[valid]
    mr = np.nansum(pv * r)
    if mr <= 0:
        return np.nan
    ratio = r / mr
    with np.errstate(divide='ignore', invalid='ignore'):
        return float(np.nansum(pv * ratio * np.log2(ratio, where=ratio > 0)))


si_rows = []
for sn, grp in clu_df.groupby('session'):
    maps, occ = session_rate_maps[sn], session_occ[sn]
    for _, row in grp.iterrows():
        si_rows.append(dict(cluster=row['cluster'], bat=row['bat'],
                            si=spatial_info(maps[row['unit']], occ)))
si_df = pd.DataFrame(si_rows).dropna(subset=['si'])
fig4i, ax = plt.subplots(figsize=(6, ph.PANEL_H + 1))
sns.boxplot(data=si_df, x='cluster', y='si', order=CLUSTER_ORDER, palette=CLUSTER_PALETTE,
            showfliers=False, ax=ax)
from scipy.stats import kruskal as _kw
groups = [si_df.loc[si_df['cluster'] == c, 'si'].values for c in CLUSTER_ORDER]
groups = [g for g in groups if len(g) > 1]
p_si = _kw(*groups).pvalue if len(groups) >= 2 else np.nan
ax.set(xlabel='cluster', ylabel='spatial info (bits/spike)', title='Fig 4i — spatial info')
ph.save_fig(fig4i, 'fig4i_spatial_info'); plt.show()
print('Fig 4i — spatial info by cluster (Kruskal-Wallis):')
ph.p_table([dict(comparison='spatial info ~ cluster', p=p_si)])


In [ ]:
# === Fig 4g: residual readouts predict CA1 vs CA3 membership (SVM) ===
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

_sub = clu_df[clu_df['subregion'].isin(['CA1', 'CA3'])].copy()
Xsub = np.stack(_sub['readout_weight_residual'].values)
ysub = (_sub['subregion'] == 'CA1').astype(int).values
clf = make_pipeline(StandardScaler(), LinearSVC(C=1.0, dual=False, max_iter=5000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
obs_bacc = cross_val_score(clf, Xsub, ysub, cv=cv, scoring='balanced_accuracy').mean()

rng = np.random.default_rng(seed)
null_bacc = np.array([
    cross_val_score(clf, Xsub, rng.permutation(ysub), cv=cv, scoring='balanced_accuracy').mean()
    for _ in range(200)])
lo, hi = ph.null_ci(null_bacc)
p_perm = (np.sum(null_bacc >= obs_bacc) + 1) / (len(null_bacc) + 1)

fig4g, ax = plt.subplots(figsize=(3.5, ph.PANEL_H + 1))
ax.axhspan(lo, hi, color='0.85', label='chance 95% CI')
ax.axhline(np.median(null_bacc), color='0.5', ls='--', lw=1)
ax.bar([0], [obs_bacc], width=0.5, color='tab:purple', edgecolor='k')
ax.set(xticks=[], ylabel='balanced accuracy', ylim=(0.3, 1.0), title='Fig 4g — CA1 vs CA3')
ph.sig_bracket(ax, -0.25, 0.25, p_perm, base=0.9)
ax.legend(frameon=False, fontsize=10)
ph.save_fig(fig4g, 'fig4g_svm'); plt.show()
print('Fig 4g — CA1/CA3 SVM balanced accuracy vs permutation null:')
ph.p_table([dict(comparison='SVM CA1 vs CA3', n_neurons=len(ysub), bal_acc=obs_bacc,
                 chance_lo=lo, chance_hi=hi, p=p_perm)])

In [ ]:
# === Fig 4j/k: attribution (neural->behaviour attention over time) ===
# The adapter exposes per-layer attention from neural (output) tokens to behaviour
# (input) tokens: A[t_out, t_in]. Averaged over heads + layers per flight.
example_attr, fp_records = None, []
with torch.no_grad():
    for sn, sd in dataloaders['test'].items():
        for b in sd:
            X = b.behavior.to(device); m = b.mask.to(device)
            _, attn = bat_former(sn, X, m, return_attn_weights=True)
            A = torch.stack([a.mean(1) for a in attn], 0).mean(0).cpu().numpy()  # (B, T, T)
            mn = b.mask.numpy().astype(bool)
            fti = b.flight_type_index.numpy()
            for i in range(A.shape[0]):
                v = mn[i]
                if v.sum() < 4:
                    continue
                Ai = A[i][np.ix_(v, v)]
                Ai = Ai / (Ai.sum(1, keepdims=True) + 1e-9)      # row-normalise
                T = Ai.shape[0]
                iu = np.triu_indices(T, k=1); il = np.tril_indices(T, k=-1)
                future = Ai[iu].sum(); past = Ai[il].sum()
                fp_records.append(dict(session=sn, bat=parse_bat(sn), cluster=int(fti[i]),
                                       fp_index=float((future - past) / (future + past + 1e-9))))
                if example_attr is None and T >= 10:
                    example_attr = (sn, int(fti[i]), Ai)

fp_df = pd.DataFrame(fp_records)
fig4jk, (axj, axk) = plt.subplots(1, 2, figsize=(11, ph.PANEL_H + 1.5),
                                  gridspec_kw=dict(width_ratios=[1, 1.4]))
if example_attr is not None:
    sn_e, cl_e, Ae = example_attr
    imj = axj.imshow(Ae, cmap='magma', aspect='auto', origin='lower')
    axj.set(title=f'Fig 4j — attribution\n{sn_e} cluster {cl_e}',
            xlabel='input time (behaviour)', ylabel='output time (neural)')
    fig4jk.colorbar(imj, ax=axj, fraction=0.046, pad=0.04)

# 4k: future-past index by flight cluster (top clusters by count).
top_clusters = fp_df['cluster'].value_counts().head(8).index.tolist()
sub = fp_df[fp_df['cluster'].isin(top_clusters)].copy()
sub['cluster'] = pd.Categorical(sub['cluster'], sorted(top_clusters), ordered=True)
sns.boxplot(data=sub, x='cluster', y='fp_index', color='0.85', showfliers=False, ax=axk)
sns.stripplot(data=sub, x='cluster', y='fp_index', hue='bat', palette=BAT_PALETTE,
              hue_order=[b for b in bats if b in set(sub['bat'])], size=3, alpha=0.5,
              ax=axk, legend=False)
axk.axhline(0, color='0.5', ls='--', lw=1)
axk.set(xlabel='flight cluster', ylabel='future − past attribution', title='Fig 4k')
ph.save_fig(fig4jk, 'fig4jk_attribution'); plt.show()
print('Fig 4k — future−past index, one-sample Wilcoxon vs 0 (per session):')
ph.p_table([dict(comparison='future-past vs 0', **ph.onesample_session_test(fp_df, 'fp_index'))])

**Figure 4 | Readout weights surface correlations in spatial and non-spatial activity.** **a**, First principal component of each session's readout-weight matrix is correlated with mean firing rate (|Pearson r| per session, coloured by bat; star: one-sample Wilcoxon vs 0). **b**, Hierarchical clustering (average linkage, cosine distance) of the firing-rate-residual readout weights across well-fit neurons: dendrogram, neuron×neuron cosine-similarity matrix (cluster colour strip), and the residual readout-weight matrix, all in cluster order. The cluster colour scheme is reused below. **c**, Example session: non-flight-rest, residual-readout, rate-map, coupling and sleep FR-correlation similarity matrices in a common (non-flight-rest clustered) order. **d**, Mantel correlations of non-flight-rest similarity with residual-readout, rate-map and coupling similarity (per session; grey = merged permutation null). **e–f**, Partial Mantel isolating the residual-readout↔non-flight relation controlling for rate map (**e**) and coupling (**f**), vs the reverse. **h**, The same Mantel comparisons split by CA1 vs CA3. **i**, Spatial information per readout cluster (Kruskal–Wallis). **g**, A linear SVM classifies CA1 vs CA3 membership from residual readouts above the permutation chance band (grey, 95%). **j**, Example neural→behaviour attention attribution matrix over time for one flight. **k**, Future−past attribution index by flight cluster (points = flights, coloured by bat). Exact *P* values are printed above each panel.

---
## Extended Data Figure 6 — cell-type and cluster comparisons

**a** pPYR fit better than pINT (FDE, corr). **b** Residual readouts less orthogonal than chance. **c** Orthogonality does not differ CA1 vs CA3. **d–i** Per readout-cluster: mean FR, FDE, corr, fraction CA1, rate-map correlation (within vs across clusters), and physical distance along the shank (within vs across). All other panels are self-contained on the joint model.

In [ ]:
# === ED Fig 6a/b/c: pPYR vs pINT fit; residual-readout orthogonality ===
fig_ed6, axes = plt.subplots(1, 4, figsize=(18, ph.PANEL_H + 1.5))

# a: pPYR vs pINT, FDE and corr (per-session medians, paired Wilcoxon).
ct = readout_df[readout_df['pType'].isin(['pPYR', 'pINT'])]
ed6_p = []
for ax, (col, lab) in zip(axes[:2], [('fde', 'FDE'), ('corr', 'corr')]):
    src = col + '_all'
    piv = ct.groupby(['session', 'pType'])[src].median().unstack('pType')
    ph.paired_lines(ax, ct.rename(columns={src: 'v'}), 'v', 'pType', ['pINT', 'pPYR'])
    r = ph.paired_session_test(ct.rename(columns={src: 'v'}), 'v', 'pType', 'pPYR', 'pINT')
    ph.sig_bracket(ax, 0, 1, r['p'])
    ax.set(ylabel=lab, title=f'ED 6a — {lab}')
    if col == 'fde':
        ax.set_ylim(-1, 1)
    ed6_p.append(dict(panel=f'a:{lab}', comparison='pPYR vs pINT', **r))

# b: residual-readout orthogonality (mean |cos|) vs chance (random unit vectors).
rng = np.random.default_rng(seed)
orth = []
for sn, grp in clu_df.groupby('session'):
    if len(grp) < 4:
        continue
    Wr = np.stack(grp['readout_weight_residual'].values)
    iu = np.triu_indices(len(grp), k=1)
    obs = np.abs((Wr @ Wr.T)[iu]).mean()
    R = rng.standard_normal((len(grp), Wr.shape[1]))
    R /= np.linalg.norm(R, axis=1, keepdims=True)
    cha = np.abs((R @ R.T)[iu]).mean()
    orth.append(dict(session=sn, bat=parse_bat(sn), observed=obs, chance=cha))
orth_df = pd.DataFrame(orth)
orth_long = orth_df.melt(['session', 'bat'], ['chance', 'observed'], 'kind', 'v')
ph.paired_lines(axes[2], orth_long, 'v', 'kind', ['chance', 'observed'])
rb = ph.paired_session_test(orth_long, 'v', 'kind', 'observed', 'chance')
ph.sig_bracket(axes[2], 0, 1, rb['p'])
axes[2].set(ylabel='mean |cos sim|', title='ED 6b — orthogonality vs chance')
ed6_p.append(dict(panel='b', comparison='observed vs chance', **rb))

# c: within-subregion orthogonality, CA1 vs CA3 (paired by session).
subo = []
for (sn, sr), grp in clu_df[clu_df['subregion'].isin(['CA1', 'CA3'])].groupby(['session', 'subregion']):
    if len(grp) < 3:
        continue
    Wr = np.stack(grp['readout_weight_residual'].values)
    iu = np.triu_indices(len(grp), k=1)
    subo.append(dict(session=sn, bat=parse_bat(sn), subregion=sr,
                     morth=np.abs((Wr @ Wr.T)[iu]).mean()))
subo_df = pd.DataFrame(subo)
ph.paired_lines(axes[3], subo_df, 'morth', 'subregion', ['CA1', 'CA3'])
rc = ph.paired_session_test(subo_df, 'morth', 'subregion', 'CA3', 'CA1')
ph.sig_bracket(axes[3], 0, 1, rc['p'])
axes[3].set(ylabel='mean |cos sim|', title='ED 6c — CA1 vs CA3 orthogonality')
ed6_p.append(dict(panel='c', comparison='CA3 vs CA1', **rc))

ph.save_fig(fig_ed6, 'edfig6abc'); plt.show()
print('ED Fig 6a-c — session-level Wilcoxon:'); ph.p_table(ed6_p)

In [ ]:
# === ED Fig 6d-i: per readout-cluster comparisons ===
from scipy.stats import kruskal, mannwhitneyu

clu_df['is_ca1'] = (clu_df['subregion'] == 'CA1').astype(float)
clu_df.loc[~clu_df['subregion'].isin(['CA1', 'CA3']), 'is_ca1'] = np.nan
cats = CLUSTER_ORDER

fig_ed6b, axes = plt.subplots(1, 5, figsize=(22, ph.PANEL_H + 1.5))
panel_specs = [('mean_fr', 'ED 6d — mean FR (Hz)'), ('fde_all', 'ED 6e — FDE'),
               ('corr_all', 'ED 6f — corr')]
ed6b_p = []
for ax, (col, title) in zip(axes[:3], panel_specs):
    sns.boxplot(data=clu_df, x='cluster', y=col, order=cats, palette=CLUSTER_PALETTE,
                showfliers=False, ax=ax)
    groups = [clu_df.loc[clu_df['cluster'] == c, col].dropna().values for c in cats]
    groups = [g for g in groups if len(g) > 1]
    p = kruskal(*groups).pvalue if len(groups) >= 2 else np.nan
    ax.set(xlabel='cluster', title=title)
    if col == 'fde_all':
        ax.set_ylim(-1, 1)
    ed6b_p.append(dict(panel=title.split('—')[0].strip(), test='Kruskal-Wallis', p=p))

# g: fraction CA1 per cluster.
frac = clu_df.groupby('cluster')['is_ca1'].mean().reindex(cats)
axes[3].bar(range(len(cats)), frac.values, color=[CLUSTER_PALETTE[c] for c in cats], edgecolor='k')
axes[3].set(xticks=range(len(cats)), xticklabels=cats, xlabel='cluster',
            ylabel='fraction CA1', title='ED 6g — frac CA1')

# i: physical distance along shank, within vs across cluster (same session+probe).
within, across = [], []
for (sn, pr), grp in clu_df.groupby(['session', 'probe_id']):
    if len(grp) < 2:
        continue
    xy = grp[['lateral', 'depth']].to_numpy()
    cl = grp['cluster'].to_numpy()
    iu, ju = np.triu_indices(len(grp), k=1)
    d = np.linalg.norm(xy[iu] - xy[ju], axis=1)
    same = cl[iu] == cl[ju]
    within.extend(d[same]); across.extend(d[~same])
dist_df = pd.DataFrame({'d': within + across,
                        'kind': ['within'] * len(within) + ['across'] * len(across)})
sns.boxplot(data=dist_df, x='kind', y='d', order=['within', 'across'],
            color='0.85', showfliers=False, ax=axes[4])
p_i = mannwhitneyu(within, across, alternative='two-sided').pvalue if within and across else np.nan
ph.sig_bracket(axes[4], 0, 1, p_i)
axes[4].set(xlabel='', ylabel='shank distance (µm)', title='ED 6i — physical distance')
ed6b_p.append(dict(panel='ED 6i', test='Mann-Whitney within vs across', p=p_i))

ph.save_fig(fig_ed6b, 'edfig6di'); plt.show()
print('ED Fig 6d-i:'); ph.p_table(ed6b_p)

In [ ]:
# === ED Fig 6h: rate-map correlation within vs across readout cluster ===
rm_within, rm_across = [], []
for sn, grp in clu_df.groupby('session'):
    if len(grp) < 2:
        continue
    units = grp['unit'].to_numpy(); cl = grp['cluster'].to_numpy()
    C = ratemap_corr_matrix(session_rate_maps[sn][units])
    iu, ju = np.triu_indices(len(units), 1)
    same = cl[iu] == cl[ju]
    vals = C[iu, ju]
    rm_within.extend(vals[same & np.isfinite(vals)])
    rm_across.extend(vals[~same & np.isfinite(vals)])
rmc_df = pd.DataFrame({'r': rm_within + rm_across,
                       'kind': ['within'] * len(rm_within) + ['across'] * len(rm_across)})
fig_ed6h, ax = plt.subplots(figsize=(4, ph.PANEL_H + 1))
sns.boxplot(data=rmc_df, x='kind', y='r', order=['within', 'across'], color='0.85',
            showfliers=False, ax=ax)
from scipy.stats import mannwhitneyu as _mwu
p_h = _mwu(rm_within, rm_across, alternative='two-sided').pvalue if rm_within and rm_across else np.nan
ph.sig_bracket(ax, 0, 1, p_h)
ax.set(xlabel='', ylabel='rate-map correlation', title='ED 6h — rate-map corr')
ph.save_fig(fig_ed6h, 'edfig6h'); plt.show()
print('ED Fig 6h — rate-map corr within vs across cluster (Mann-Whitney):')
ph.p_table([dict(comparison='within vs across', p=p_h)])


**Extended Data Figure 6 | Cell-type and readout-cluster comparisons.** **a**, Putative pyramidal cells (pPYR) are better fit than putative interneurons (pINT) by FDE and correlation (per-session medians, paired Wilcoxon). **b**, Residual readout weights are less orthogonal (higher within-session mean |cosine similarity|) than random unit vectors of matched dimension. **c**, Orthogonality does not differ between CA1 and CA3 (paired by session). **d–f**, Mean firing rate, FDE and correlation per readout cluster (Kruskal–Wallis across clusters). **g**, Fraction of CA1 neurons per cluster. **h**, Rate-map correlation is higher within than across clusters (Mann–Whitney). **i**, Physical distance along the shank is smaller within than across clusters (Mann–Whitney). Exact *P* values printed above; FDE axes clipped to [−1, 1] for display.

---
## Figure 5 — SWRs are associated with greater flight novelty and less predictable hippocampal activity

**a** SWRs elevated in a 3 s window before takeoff and after landing. **b** SWR rate around non-perch/non-reward ("other") flight edges vs perch and reward. **c** Rarer (unique) flight types are preceded/followed by more SWRs. **d** SWR rate vs cluster size (per-session slopes). **e** Overall SWR rate is stable across the session while peri-flight SWRs decline. **f–g** Increased peri-flight SWRs predict worse encoding (lower FDE) and decoding (higher log Fréchet), beyond session-time and flight-novelty effects (mixed model).

> SWR events are loaded directly from `RPL_converted/RPL_probe*.mat` and flight bouts from `Extracted_Behavior_*.mat` (raw data on the mount) — independent of the shared dataloaders. Restricted to the 17 `DEFAULT_SESSIONS` (`Dataset_3_14543_240419` is excluded for consistency even though it has ripple data).

In [ ]:
def edge_class(at_perch, at_rew):
    """Flight-endpoint class from proximity masks: perch / reward / other."""
    return np.where(at_perch, 'perch', np.where(at_rew, 'reward', 'other'))

In [ ]:
# === Figure 5 foundation: SWR events + reconstructed per-flight table (w/ clusters) ===
import glob, io, contextlib
from scipy.io import loadmat
import digitalbat.datasets.data as _ddata
from digitalbat.datasets.defaults import DEFAULT_PARENT_FOLDER
from notebook_helpers import compute_target_proximity

PARENT = DEFAULT_PARENT_FOLDER.rstrip('/')
PSTH_WIN, W_PERI = 30.0, 3.0


def load_flight_bouts(sess):
    bhv = loadmat(glob.glob(f'{PARENT}/{sess}/Extracted_Behavior_*.mat')[0])
    fly = bhv['bflying'][:, 0] > 0
    t = bhv['t'][:, 0] if 'Dataset_2' in sess else bhv['t'][0, :]
    d = np.diff(fly.astype(np.int8))
    starts = np.where(d == 1)[0] + 1
    stops = np.where(d == -1)[0] + 1
    if fly[0]:
        starts = np.r_[0, starts]
    if fly[-1]:
        stops = np.r_[stops, len(fly)]
    return t[starts], t[stops - 1]


def load_swr_events(sess):
    """Merged (across-probe) ripple event times (s)."""
    t_all, dur_all = [], []
    for f in sorted(glob.glob(f'{PARENT}/RPL_converted/{sess}/RPL_probe*.mat')):
        tbl = np.atleast_1d(loadmat(f, squeeze_me=True, struct_as_record=True)['RPL_out']['table'].item())
        if tbl.size == 0:
            continue
        t_all.append(np.asarray(tbl['t'], dtype=float).ravel())
        dur_all.append(np.asarray(tbl['dur'], dtype=float).ravel())
    if not t_all:
        return np.zeros(0)
    t, dur = np.concatenate(t_all), np.concatenate(dur_all)
    order = np.argsort(t)
    s, e = t[order], (t + dur)[order]
    ms, me = [s[0]], [e[0]]
    for i in range(1, len(s)):
        if s[i] <= me[-1]:
            me[-1] = max(me[-1], e[i])
        else:
            ms.append(s[i]); me.append(e[i])
    return (np.array(ms) + np.array(me)) / 2


def peri_event_diffs(edges, swr_t, window):
    out = [swr_t[np.abs(swr_t - et) <= window] - et for et in edges]
    return np.concatenate(out) if out else np.zeros(0)


def reconstruct_flights(sess):
    """Re-run Data_Prep to recover per-flight cluster id + start/stop times +
    takeoff/landing positions (aligned to the joint model's flight ordering)."""
    folder = f'{PARENT}/{sess}'
    su = f'{PARENT}/To_transfer/out_Gunits_probe*_{sess}_with_metrics.mat'
    dp = _ddata.Data_Prep_for_Raw(
        parent_folder=folder, su_file_path=su, only_flight_flag=True,
        use_speed_head_dir=False, use_IMU_flag=False, use_click_flag=False,
        use_LFP_flag=False, use_ripple_flag=False, spike_count_flag=True,
        bin_width=bin_width, max_acceleration=50)
    with contextlib.redirect_stdout(io.StringIO()):
        dp.get_all_behavior(); dp.get_all_spike(); dp.get_flight_cluster()
    bhv = loadmat(glob.glob(folder + '/Extracted_Behavior_*.mat')[0])
    bt = bhv['t'][:, 0] if 'Dataset_2' in sess else bhv['t'][0, :]
    fcr, atf = dp.flight_cluster_results, dp.all_time_flight
    pre = []
    for cid in np.unique(fcr['id']):
        sel = np.where(fcr['id'] == cid)[0]
        st, sp = bt[fcr['start_frame'][sel]], bt[fcr['stop_frame'][sel]]
        for ii in range(len(st)):
            ind = np.where((atf >= st[ii]) & (atf <= sp[ii]))[0]
            if len(ind):
                pos = dp.all_behavior[:3, ind].T
                acc = np.gradient(np.gradient(pos, bin_width, axis=0), bin_width, axis=0)
                pre.append((int(cid), float(st[ii]), float(sp[ii]), pos[0], pos[-1],
                            float(np.linalg.norm(acc, axis=1).max())))
    kept = [r for r in pre if r[5] <= 50]
    rows = []
    for fpi, (cid, st, sp, toff, land, _) in enumerate(kept):
        rows.append(dict(session=sess, bat=parse_bat(sess), flight_path_index=fpi,
                         flight_type_index=cid, start_t=st, stop_t=sp, dur=sp - st,
                         toff=toff, land=land))
    return pd.DataFrame(rows)


swr_times_by_session, peri_rows, flight_parts = {}, [], []
for sess in session_names:                       # 17 DEFAULT_SESSIONS
    if not (glob.glob(f'{PARENT}/{sess}/Extracted_Behavior_*.mat')
            and os.path.isdir(f'{PARENT}/RPL_converted/{sess}')):
        print(f'skip {sess}: missing behaviour/ripple'); continue
    swr_t = np.sort(load_swr_events(sess))
    if len(swr_t) == 0:
        continue
    swr_times_by_session[sess] = swr_t
    start_t, stop_t = load_flight_bouts(sess)
    for align, edges in (('onset', start_t), ('offset', stop_t)):
        peri_rows.append(pd.DataFrame(dict(session=sess, bat=parse_bat(sess), align=align,
                                           dt=peri_event_diffs(edges, swr_t, PSTH_WIN))))
    fdf = reconstruct_flights(sess)
    for kind in ('toff', 'land'):
        prox = compute_target_proximity(np.vstack(fdf[kind].values), session_name=sess)
        fdf[f'{kind}_class'] = edge_class(prox['at_perch'], prox['at_rew'])
    fdf['pre_takeoff'] = np.searchsorted(swr_t, fdf['start_t']) - np.searchsorted(swr_t, fdf['start_t'] - W_PERI)
    fdf['post_landing'] = np.searchsorted(swr_t, fdf['stop_t'] + W_PERI) - np.searchsorted(swr_t, fdf['stop_t'])
    flight_parts.append(fdf)
    print(f'{sess}: {len(fdf)} flights, {len(swr_t)} SWR')

peri_df = pd.concat(peri_rows, ignore_index=True)
flights = pd.concat(flight_parts, ignore_index=True)
flights['pre_rate'] = flights['pre_takeoff'] / W_PERI
flights['post_rate'] = flights['post_landing'] / W_PERI
flights['cluster_size'] = flights.groupby(['session', 'flight_type_index'])['flight_path_index'].transform('size')
flights['repeated'] = np.where(flights['cluster_size'] > 1, 'repeated', 'unique')
flights = flights.sort_values(['session', 'start_t'])
flights['flight_num'] = flights.groupby('session').cumcount()
flights['flight_frac'] = flights.groupby('session')['flight_num'].transform(lambda s: s / max(s.max(), 1))
print(f'\n{len(flights)} flights, {flights.session.nunique()} sessions, {flights.bat.nunique()} bats')

In [ ]:
# === Fig 5a: peri-flight SWR rate time course; 5b: by endpoint class ===
from scipy.stats import friedmanchisquare

n_flights_total = len(flights)
BW = 1.0
bins = np.arange(-PSTH_WIN, PSTH_WIN + BW, BW); ctr = bins[:-1] + BW / 2

fig5ab = plt.figure(figsize=(16, ph.PANEL_H + 1.5))
gs = fig5ab.add_gridspec(1, 3, width_ratios=[1, 1, 1])
for k, (align, lab) in enumerate([('onset', 'takeoff'), ('offset', 'landing')]):
    ax = fig5ab.add_subplot(gs[k])
    dt = peri_df.loc[peri_df['align'] == align, 'dt'].values
    rate = np.histogram(dt, bins=bins)[0] / (n_flights_total * BW)
    ax.bar(ctr, rate, width=BW, color='steelblue', alpha=0.85)
    ax.axvline(0, color='crimson', ls='--', lw=1.5)
    ax.set(xlabel=f'time from {lab} (s)', title=f'Fig 5a — {lab}',
           ylabel='SWR rate (events/flight/s)' if k == 0 else '')

# 5b: per-session mean peri-flight SWR rate by endpoint class (perch/reward/other).
edges = pd.concat([
    flights[['session', 'bat', 'pre_rate', 'toff_class']].rename(
        columns={'pre_rate': 'rate', 'toff_class': 'cls'}),
    flights[['session', 'bat', 'post_rate', 'land_class']].rename(
        columns={'post_rate': 'rate', 'land_class': 'cls'})], ignore_index=True)
CLS3 = ['perch', 'reward', 'other']
ps = edges.groupby(['session', 'bat', 'cls'])['rate'].mean().reset_index()
ps['cls'] = pd.Categorical(ps['cls'], CLS3, ordered=True)
axb = fig5ab.add_subplot(gs[2])
sns.boxplot(data=ps, x='cls', y='rate', order=CLS3, color='0.85', showfliers=False, ax=axb)
sns.stripplot(data=ps, x='cls', y='rate', order=CLS3, hue='bat', palette=BAT_PALETTE,
              hue_order=[b for b in bats if b in set(ps['bat'])], size=7, jitter=0.12,
              edgecolor='k', linewidth=0.4, ax=axb, legend=False)
wide = ps.pivot_table(index='session', columns='cls', values='rate', observed=False).dropna()
p_omni = friedmanchisquare(*[wide[c].values for c in CLS3])[1] if len(wide) >= 3 else np.nan
b5_p = [dict(comparison='Friedman perch/reward/other', n=len(wide), p=p_omni)]
lvl = 0
for a, b in [(0, 2), (1, 2), (0, 1)]:
    r = ph.paired_session_test(ps.rename(columns={'rate': 'v', 'cls': 'c'}), 'v', 'c', CLS3[a], CLS3[b])
    ph.sig_bracket(axb, a, b, r['p'], level=lvl); lvl += 1
    b5_p.append(dict(comparison=f'{CLS3[a]} vs {CLS3[b]}', **r))
axb.set(xlabel='', ylabel='SWR rate (Hz)', title='Fig 5b — by endpoint class')
ph.save_fig(fig5ab, 'fig5ab'); plt.show()
print('Fig 5b — endpoint class comparisons:'); ph.p_table(b5_p)

In [ ]:
# === Fig 5c: repeated vs unique; 5d: cluster-size slopes; 5e: session dynamics ===
fig5cde, axes = plt.subplots(1, 4, figsize=(20, ph.PANEL_H + 1.5))
c5_p = []

# 5c: repeated vs unique flights, peri-flight SWR rate (before takeoff shown).
psc = flights.groupby(['session', 'bat', 'repeated'])['pre_rate'].mean().reset_index()
ph.paired_lines(axes[0], psc.rename(columns={'pre_rate': 'v'}), 'v', 'repeated', ['unique', 'repeated'])
rc = ph.paired_session_test(psc.rename(columns={'pre_rate': 'v'}), 'v', 'repeated', 'unique', 'repeated')
ph.sig_bracket(axes[0], 0, 1, rc['p'])
axes[0].set(ylabel='pre-takeoff SWR rate (Hz)', title='Fig 5c — flight novelty')
c5_p.append(dict(comparison='unique vs repeated (pre)', **rc))

# 5d: per-session slope of pre-takeoff SWR rate vs cluster size, tested vs 0.
slopes = []
for sn, d in flights.groupby('session'):
    if d['cluster_size'].nunique() < 2 or len(d) < 8:
        continue
    b = np.polyfit(d['cluster_size'].values, d['pre_rate'].values, 1)[0]
    slopes.append(dict(session=sn, bat=parse_bat(sn), slope=b))
slope_df = pd.DataFrame(slopes)
sns.stripplot(data=slope_df, y='slope', hue='bat', palette=BAT_PALETTE,
              hue_order=[b for b in bats if b in set(slope_df['bat'])], size=8, jitter=0.15,
              edgecolor='k', linewidth=0.4, ax=axes[1], legend=False)
axes[1].axhline(0, color='0.5', ls='--', lw=1)
rd = ph.onesample_session_test(slope_df, 'slope', popmean=0.0)
ph.sig_bracket(axes[1], -0.3, 0.3, rd['p'])
axes[1].set(xticks=[], ylabel='slope: SWR rate vs cluster size', title='Fig 5d — cluster-size slope')
c5_p.append(dict(comparison='cluster-size slope vs 0', **rd))

# 5e: overall SWR rate over session (deciles) — per-session lines + mean 95% CI.
NBIN = 10
curves = []
for sn, t in swr_times_by_session.items():
    if len(t) < 3 * NBIN:
        continue
    e = np.linspace(t.min(), t.max(), NBIN + 1)
    r = np.histogram(t, bins=e)[0] / ((t.max() - t.min()) / NBIN)
    curves.append((parse_bat(sn), r / r.mean()))
xb = (np.arange(NBIN) + 0.5) / NBIN
for bt, c in curves:
    axes[2].plot(xb, c, color=BAT_PALETTE.get(bt, '0.6'), lw=0.8, alpha=0.35)
C = np.array([c for _, c in curves])
mu = C.mean(0); ci = 1.96 * C.std(0) / np.sqrt(len(C))
axes[2].plot(xb, mu, color='k', lw=2.5)
axes[2].fill_between(xb, mu - ci, mu + ci, color='k', alpha=0.2)
axes[2].axhline(1, color='0.5', ls=':', lw=1)
axes[2].set(xlabel='normalized session time', ylabel='SWR rate (/ session mean)',
            title='Fig 5e — overall rate (stable)')

# 5e (peri-flight): peri-flight SWR rate vs normalized flight number (declines).
fb = np.linspace(0, 1, 9)
idx = np.clip(np.digitize(flights['flight_frac'].values, fb) - 1, 0, len(fb) - 2)
for col, lab, cc in [('pre_rate', 'before takeoff', 'tab:blue'), ('post_rate', 'after landing', 'tab:orange')]:
    v = flights[col].values
    mu = [v[idx == k].mean() for k in range(len(fb) - 1)]
    se = [v[idx == k].std() / np.sqrt(max((idx == k).sum(), 1)) for k in range(len(fb) - 1)]
    axes[3].errorbar((fb[:-1] + fb[1:]) / 2, mu, yerr=1.96 * np.array(se), fmt='o-', color=cc, label=lab, capsize=3)
axes[3].legend(frameon=False, fontsize=10)
axes[3].set(xlabel='normalized flight number', ylabel='peri-flight SWR rate (Hz)',
            title='Fig 5e — peri-flight (declines)')
ph.save_fig(fig5cde, 'fig5cde'); plt.show()
print('Fig 5c/d:'); ph.p_table(c5_p)

In [ ]:
# === Fig 5f/g: peri-flight SWRs predict worse encoding (FDE) + decoding (log Fréchet) ===
# Mixed model holding session, flight cluster and normalized session-time fixed.
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
from digitalbat.utility.utils import poisson_fraction_deviance_explained


@torch.no_grad()
def joint_encode_perflight():
    rows = []
    for sn, sd in joint_loaders['test'].items():
        for b in sd:
            m = b.mask.to(device)
            rate, _, _ = joint_model(sn, b.spike.to(device), b.behavior.to(device),
                                     m, m, torch.zeros_like(m))
            rate = rate.cpu().numpy(); yt = b.spike.numpy(); mn = b.mask.numpy().astype(bool)
            fpi = b.flight_path_index.numpy()
            for i in range(rate.shape[0]):
                v = mn[i]
                if v.sum() < 2:
                    continue
                rows.append(dict(session=sn, flight_path_index=int(fpi[i]),
                                 fde=float(poisson_fraction_deviance_explained(yt[i, v], rate[i, v]))))
    return pd.DataFrame(rows)


perflight_fde = joint_encode_perflight()
perflight_fr = joint_frechet.rename(columns={'flight_id': 'flight_path_index'})[
    ['session', 'flight_path_index', 'frechet_3d']]
mf = (flights.merge(perflight_fde, on=['session', 'flight_path_index'], how='inner')
             .merge(perflight_fr, on=['session', 'flight_path_index'], how='inner'))
mf['log_frechet'] = np.log(mf['frechet_3d'].clip(lower=1e-3))
print(f'Fig 5f/g merged flights: {len(mf)} (of {len(flights)})')


def added_variable_panel(ax, df, outcome, ylabel, title):
    """Mixed model outcome ~ pre_takeoff + flight_frac + (1|session) + cluster FE;
    plot the added-variable (partial-residual) relationship vs residual #SWRs."""
    d = df.dropna(subset=[outcome, 'pre_takeoff', 'flight_frac']).copy()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        lme = smf.mixedlm(f'{outcome} ~ pre_takeoff + flight_frac',
                          d, groups=d['session'],
                          vc_formula={'cluster': '0 + C(flight_type_index)'}).fit(method='lbfgs', maxiter=400)
    beta, p = lme.params['pre_takeoff'], lme.pvalues['pre_takeoff']
    # added-variable residuals: regress out flight_frac + session from both terms.
    Xc = sm.add_constant(pd.get_dummies(d['session'], drop_first=True).astype(float)
                         .assign(flight_frac=d['flight_frac'].values))
    ry = sm.OLS(d[outcome].values, Xc).fit().resid
    rx = sm.OLS(d['pre_takeoff'].values, Xc).fit().resid
    ax.scatter(rx, ry, s=8, alpha=0.25, color='0.5', rasterized=True)
    coef = np.polyfit(rx, ry, 1)
    xs = np.linspace(rx.min(), rx.max(), 50)
    ax.plot(xs, np.polyval(coef, xs), color='firebrick', lw=2.5)
    ax.set(xlabel='residual # peri-flight SWRs', ylabel=ylabel, title=title)
    return dict(comparison=title, beta=float(beta), p=float(p), n=len(d))


fig5fg, (axf, axg) = plt.subplots(1, 2, figsize=(11, ph.PANEL_H + 1.5))
r_f = added_variable_panel(axf, mf, 'fde', 'residual FDE', 'Fig 5f — encoding')
r_g = added_variable_panel(axg, mf, 'log_frechet', 'residual log Fréchet', 'Fig 5g — decoding')
ph.save_fig(fig5fg, 'fig5fg'); plt.show()
print('Fig 5f/g — mixed model (# peri-flight SWRs | session, cluster, session-time):')
ph.p_table([r_f, r_g])

**Figure 5 | SWRs are associated with greater flight novelty and less predictable hippocampal activity.** **a**, Peri-flight SWR rate (events per flight per second, pooled) rises in the seconds before takeoff and after landing. **b**, Peri-flight SWR rate by flight-endpoint class: elevated around non-perch, non-reward ("other") edges relative to perch and reward (per-session means, coloured by bat; Friedman omnibus + pairwise Wilcoxon). **c**, Rarer (unique) flight clusters are preceded by more SWRs than repeated clusters (paired by session). **d**, Per-session slope of pre-takeoff SWR rate against cluster size is negative (one-sample Wilcoxon vs 0). **e**, Overall SWR rate is stable across the session (thin lines = sessions coloured by bat; black = mean ± 95% CI) while peri-flight SWR rate declines with flight number. **f–g**, Added-variable plots from a linear mixed model (`outcome ~ #peri-flight SWRs + normalized session time`, random intercepts for session, cluster fixed effects): more peri-flight SWRs predict lower encoding FDE (**f**) and higher log Fréchet decoding error (**g**). Exact *P* values (mixed-model coefficient for #SWRs) printed above.

---
## Extended Data Figure 3 — foundation-model transfer

**a** 17-fold leave-one-session-out CV: fine-tuning the foundation model (core at 0.1× learning rate) vs training the same architecture entirely on the held-out session (scratch). **b** As **a**, for 6-fold leave-one-bat-out CV. Loaded from the cached transfer-CV result tables (`out/transfer/*/results_*.pkl`).

In [ ]:
# === ED Fig 3: foundation-transfer CV (fine-tune vs scratch), FDE ===
TRANSFER_ROOT = './out/transfer'


def load_transfer_group(g):
    parts = []
    for suffix in ('', '_lowlr'):
        p = f'{TRANSFER_ROOT}/{g}/results_{g}{suffix}.pkl'
        if os.path.exists(p):
            parts.append(pd.read_pickle(p))
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    df['mode'] = df['mode'].replace({'finetune': 'frozen'})
    return df


# Fine-tune mode: prefer 'lowlr' (core at 0.1x LR); fall back to 'frozen'.
GROUPS = [('session', 'ED 3a — 17-fold (leave-one-session-out)'),
          ('bat', 'ED 3b — 6-fold (leave-one-bat-out)')]
fig_ed3, axes = plt.subplots(1, 2, figsize=(11, ph.PANEL_H + 1.5))
ed3_p = []
for ax, (grp, title) in zip(axes, GROUPS):
    df = load_transfer_group(grp)
    if df.empty:
        ax.set_visible(False); continue
    ft_mode = 'lowlr' if 'lowlr' in set(df['mode']) else 'frozen'
    sub = df[df['mode'].isin([ft_mode, 'scratch'])].copy()
    # per-fold (target session) median FDE, paired fine-tune vs scratch.
    per = (sub.groupby(['target_session', 'mode'])['fde'].median().unstack('mode')
              .dropna(subset=[ft_mode, 'scratch']).reset_index())
    per['bat'] = per['target_session'].map(parse_bat)
    long = per.melt(['target_session', 'bat'], [ft_mode, 'scratch'], 'mode', 'fde')
    long['mode'] = long['mode'].replace({ft_mode: 'fine-tune'})
    piv = ph.paired_lines(ax, long, 'fde', 'mode', ['scratch', 'fine-tune'],
                          session_col='target_session')
    res = ph.paired_session_test(long, 'fde', 'mode', 'fine-tune', 'scratch',
                                 session_col='target_session')
    ph.sig_bracket(ax, 0, 1, res['p'])
    ax.set(ylabel='held-out FDE', title=title, ylim=(-1, 1))
    ed3_p.append(dict(panel=title.split('—')[0].strip(), comparison='fine-tune vs scratch', **res))
ph.bat_legend(axes[0], loc='lower right')
ph.save_fig(fig_ed3, 'edfig3_transfer'); plt.show()
print('ED Fig 3 — fine-tune vs scratch (paired per fold, Wilcoxon):'); ph.p_table(ed3_p)

---
## Extended Data Figure 5 — neural PCA vs latent projection

**a** PCA applied directly to neural activity fails to reveal route structure (single session). **b** Neural activity projected into the 3-D transformer-latent embedding via ridge regression (leave-one-flight-out CV) recovers the route structure (same session).

In [ ]:
# === ED Fig 5: neural PCA vs ridge-projection into latent space (one session) ===
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import silhouette_score

# Pick the indoor session with the most test flights.
ed5_session = max((s for s in session_names if ph.is_indoor(s)),
                  key=lambda s: len(np.unique(emb['flight_id'][emb['session'] == s])))
sel = emb['session'] == ed5_session
lat_s, route_s, fid_s = emb['latent'][sel], emb['route'][sel], emb['flight_id'][sel]

# Neural activity (spike counts) for the same session/timepoints, in dataloader order.
spk_rows = []
for b in dataloaders['test'][ed5_session]:
    mn = b.mask.numpy().astype(bool)
    spk = b.spike.numpy()
    for i in range(spk.shape[0]):
        if mn[i].sum() > 0:
            spk_rows.append(spk[i, mn[i]])
spk_s = np.vstack(spk_rows)
assert len(spk_s) == len(lat_s), (len(spk_s), len(lat_s))

# a: direct 3D PCA of neural activity (sqrt-transformed counts).
neural_pca = PCA(n_components=3, random_state=seed).fit_transform(np.sqrt(spk_s))
# b: ridge (leave-one-flight-out CV) predicting the 3D latent-PCA from neural activity.
lat_pca = PCA(n_components=3, random_state=seed).fit_transform(lat_s)
pred = np.zeros_like(lat_pca)
for tr, te in LeaveOneGroupOut().split(spk_s, groups=fid_s):
    pred[te] = Ridge(alpha=10.0).fit(np.sqrt(spk_s[tr]), lat_pca[tr]).predict(np.sqrt(spk_s[te]))

fig_ed5, (axa, axb) = plt.subplots(1, 2, figsize=(11, 5))
ph.plot_embedding(axa, neural_pca[:, :2], route_s, categorical=True, s=5)
axa.set_title('ED 5a — neural PCA (direct)')
ph.plot_embedding(axb, pred[:, :2], route_s, categorical=True, s=5)
axb.set_title('ED 5b — ridge → latent (LOFO CV)')
handles = [plt.Line2D([], [], marker='o', ls='', color=ph.ROUTE_COLORS[k], label=ph.ROUTE_LABELS[k])
           for k in range(len(ph.ROUTE_LABELS))]
axb.legend(handles=handles, title='route', loc='center left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=10)
fig_ed5.suptitle(f'ED Fig 5 — {ed5_session}', y=1.02)
ph.save_fig(fig_ed5, 'edfig5_neural_pca'); plt.show()

# Quantify structure: route-label silhouette in each space (higher = clearer).
mask_r = route_s > 0
sil_a = silhouette_score(neural_pca[mask_r], route_s[mask_r]) if mask_r.sum() > 10 else np.nan
sil_b = silhouette_score(pred[mask_r], route_s[mask_r]) if mask_r.sum() > 10 else np.nan
print(f'route silhouette — neural PCA: {sil_a:.3f} | ridge→latent: {sil_b:.3f}')

**Extended Data Figure 5 | Neural PCA versus latent projection.** For one representative indoor session, each point is a held-out flight time-bin coloured by flight route. **a**, The top two principal components of the (√-transformed) neural activity do not separate routes. **b**, Predicting the 3-D transformer-latent PCA from the same neural activity by ridge regression under leave-one-flight-out cross-validation recovers clear route structure. The route-label silhouette scores (printed) quantify the difference.

---
## Extended Data Figure 7 — SWR mixed-model ladders

**a** Non-reward peri-flight SWR counts (perch + other). **b** Mixed model of decoding error vs number of flights in the cluster (| bat). **c** Full mixed-model ladder (naïve pooled → per-session lines → added-variable) for encoding FDE and correlation. **d** Full ladder for decoding (3-D log Fréchet and 3-D reconstruction error).

In [ ]:
# === ED Fig 7: SWR mixed-model ladders ===
# Extend the Fig 5f/g merged table with per-flight corr (encode) and recon (decode).
@torch.no_grad()
def joint_encode_perflight_corr():
    rows = []
    for sn, sd in joint_loaders['test'].items():
        for b in sd:
            m = b.mask.to(device)
            rate, _, _ = joint_model(sn, b.spike.to(device), b.behavior.to(device),
                                     m, m, torch.zeros_like(m))
            rate = rate.cpu().numpy(); yt = b.spike.numpy(); mn = b.mask.numpy().astype(bool)
            fpi = b.flight_path_index.numpy()
            for i in range(rate.shape[0]):
                v = mn[i]
                if v.sum() < 3:
                    continue
                c = corr(yt[i, v], rate[i, v], axis=0)
                rows.append(dict(session=sn, flight_path_index=int(fpi[i]),
                                 corr=float(np.nanmean(c))))
    return pd.DataFrame(rows)


recon_fl = (joint_decode_test.groupby(['session', 'flight_id'])['err_3d'].median()
            .reset_index().rename(columns={'flight_id': 'flight_path_index', 'err_3d': 'recon_3d'}))
mf2 = (mf.merge(joint_encode_perflight_corr(), on=['session', 'flight_path_index'], how='left')
         .merge(recon_fl, on=['session', 'flight_path_index'], how='left'))


def lme_p(df, outcome, x):
    d = df.dropna(subset=[outcome, x, 'flight_frac'])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        lme = smf.mixedlm(f'{outcome} ~ {x} + flight_frac', d, groups=d['session'],
                          vc_formula={'cluster': '0 + C(flight_type_index)'}).fit(method='lbfgs', maxiter=400)
    return float(lme.params[x]), float(lme.pvalues[x]), len(d)


def ladder_row(axes3, df, outcome, ylabel, x='pre_takeoff'):
    d = df.dropna(subset=[outcome, x, 'flight_frac']).copy()
    # (1) naive pooled
    axes3[0].scatter(d[x], d[outcome], s=6, alpha=0.15, color='0.5', rasterized=True)
    cf = np.polyfit(d[x], d[outcome], 1); xs = np.linspace(d[x].min(), d[x].max(), 50)
    axes3[0].plot(xs, np.polyval(cf, xs), color='firebrick', lw=2)
    axes3[0].set(xlabel=f'# peri-flight SWRs', ylabel=ylabel, title='naïve pooled')
    # (2) per-session lines coloured by bat
    for sn, g in d.groupby('session'):
        if len(g) < 8 or g[x].nunique() < 2:
            continue
        cfs = np.polyfit(g[x], g[outcome], 1); xg = np.linspace(g[x].min(), g[x].max(), 20)
        axes3[1].plot(xg, np.polyval(cfs, xg), color=BAT_PALETTE.get(parse_bat(sn), '0.6'), lw=1, alpha=0.7)
    axes3[1].set(xlabel='# peri-flight SWRs', ylabel=ylabel, title='per-session')
    # (3) added-variable + LME p
    Xc = sm.add_constant(pd.get_dummies(d['session'], drop_first=True).astype(float)
                         .assign(flight_frac=d['flight_frac'].values))
    ry = sm.OLS(d[outcome].values, Xc).fit().resid
    rx = sm.OLS(d[x].values, Xc).fit().resid
    axes3[2].scatter(rx, ry, s=6, alpha=0.15, color='0.5', rasterized=True)
    cf2 = np.polyfit(rx, ry, 1); xs2 = np.linspace(rx.min(), rx.max(), 50)
    axes3[2].plot(xs2, np.polyval(cf2, xs2), color='firebrick', lw=2)
    beta, p, n = lme_p(d, outcome, x)
    axes3[2].set(xlabel='residual # SWRs', ylabel=f'residual {ylabel}', title='added-variable')
    return dict(outcome=ylabel, beta=beta, p=p, n=n)


# 7a: non-reward peri-flight SWR counts (perch + other).
fig7a, ax7a = plt.subplots(figsize=(5, ph.PANEL_H + 1))
nonrew = pd.concat([
    flights.loc[flights['toff_class'] != 'reward', 'pre_takeoff'],
    flights.loc[flights['land_class'] != 'reward', 'post_landing']])
ax7a.hist(nonrew, bins=np.arange(0, nonrew.max() + 2) - 0.5, color='steelblue', edgecolor='k')
ax7a.set(xlabel='# peri-flight SWRs (non-reward edges)', ylabel='count', title='ED 7a')
ph.save_fig(fig7a, 'edfig7a'); plt.show()

# 7b: decode error vs #flights in cluster (| bat) — added-variable panel.
fig7b, ax7b = plt.subplots(figsize=(5, ph.PANEL_H + 1))
d7b = mf2.dropna(subset=['log_frechet', 'cluster_size']).copy()
Xc = sm.add_constant(pd.get_dummies(d7b['bat'], drop_first=True).astype(float))
ry = sm.OLS(d7b['log_frechet'].values, Xc).fit().resid
rx = sm.OLS(d7b['cluster_size'].values.astype(float), Xc).fit().resid
ax7b.scatter(rx, ry, s=6, alpha=0.2, color='0.5', rasterized=True)
cf = np.polyfit(rx, ry, 1); xs = np.linspace(rx.min(), rx.max(), 50)
ax7b.plot(xs, np.polyval(cf, xs), color='firebrick', lw=2)
beta_b, p_b, n_b = lme_p(d7b.assign(pre_takeoff=d7b['cluster_size']), 'log_frechet', 'cluster_size')
ax7b.set(xlabel='residual # flights in cluster', ylabel='residual log Fréchet', title='ED 7b')
ph.save_fig(fig7b, 'edfig7b'); plt.show()

# 7c: encoding ladders (FDE, corr); 7d: decoding ladders (log Fréchet, recon).
ed7_p = [dict(panel='7b', outcome='log Fréchet ~ cluster size', beta=beta_b, p=p_b, n=n_b)]
for tag, rows in [('ED 7c', [('fde', 'FDE'), ('corr', 'corr')]),
                  ('ED 7d', [('log_frechet', 'log Fréchet'), ('recon_3d', '3D recon (m)')])]:
    figL, axesL = plt.subplots(len(rows), 3, figsize=(15, 4.2 * len(rows)), squeeze=False)
    for r, (col, lab) in enumerate(rows):
        rec = ladder_row(axesL[r], mf2, col, lab)
        ed7_p.append(dict(panel=tag, **rec))
    figL.suptitle(f'{tag} — mixed-model ladder', y=1.01)
    ph.save_fig(figL, f'edfig7{"c" if tag.endswith("c") else "d"}'); plt.show()

print('ED Fig 7 — mixed-model coefficients for #SWRs / cluster size:'); ph.p_table(ed7_p)

---
## Extended Data Figure 4 — context window size

**a–b** Effect of the training context-window size on encoding fit (FDE, correlation): models retrained at window sizes {1, 3, 5, 7, 9, 11, full} with `window_size=W` flight tiling, evaluated on the matching windowed test set. **c–d** As **a–b** for decoding (3-D Fréchet, 3-D reconstruction). Panels **c–d** are scaffolded — the context-length decode sweep trains a decoder per window size (models not among the cached encoder checkpoints).

In [ ]:
# === ED Fig 4a/b: encoding fit vs training context-window size ===
CTX_DIR = './out/bat_former_ctxlen'
CTX_WINDOWS = [1, 3, 5, 7, 9, 11]


@torch.no_grad()
def eval_windowed(model, loaders):
    rows = []
    for sn, sd in loaders['test'].items():
        yv, yp = [], []
        for b in sd:
            m = b.mask.to(device)
            out = model(sn, b.behavior.to(device), m)
            v = m == 1.0
            yv.append(b.spike.to(device)[v].cpu().numpy()); yp.append(out[v].cpu().numpy())
        yv, yp = np.vstack(yv), np.vstack(yp)
        c, f = corr(yv, yp, axis=0), poisson_fde_population(yv, yp)
        for u in range(yv.shape[1]):
            rows.append(dict(session=sn, bat=parse_bat(sn), unit=u, corr=float(c[u]), fde=float(f[u])))
    return pd.DataFrame(rows)


ctx_pieces = []
for W in CTX_WINDOWS:
    ck = f'{CTX_DIR}/bat_former_seed_{seed}_lr_1.00e-02_wd_1.00e-02{DATA_TAG}_ctx{W}.pth'
    if not os.path.exists(ck):
        print(f'  [skip ctx{W}] no checkpoint'); continue
    dlW, _, _, _ = bat_data_loader(**{**data_config, 'window_size': W,
                                      'stride_train': W, 'stride_test': W})
    set_random_state(seed)
    mW = build_former('transformer').to(device)
    mW.load_state_dict(torch.load(ck, map_location=device))
    ctx_pieces.append(eval_windowed(mW.eval(), dlW).assign(context=W, context_x=W))
    print(f'  ok: ctx{W}')
# 'full' context = the main transformer on full-flight loaders.
ctx_pieces.append(ed1_encode_df[ed1_encode_df['model'] == 'Transformer']
                  [['session', 'bat', 'unit', 'corr', 'fde']].assign(context='full', context_x=13))
ctx_df = pd.concat(ctx_pieces, ignore_index=True)

# Per-session median vs context, mean +/- 95% CI across sessions.
fig_ed4, axes = plt.subplots(1, 2, figsize=(11, ph.PANEL_H + 1))
for ax, (col, lab, ylim) in zip(axes, [('fde', 'FDE', (-0.25, 0.6)), ('corr', 'corr', (0, 0.7))]):
    per = ctx_df.groupby(['context_x', 'session'])[col].median().reset_index()
    for sn, g in per.groupby('session'):
        g = g.sort_values('context_x')
        ax.plot(g['context_x'], g[col], color=BAT_PALETTE.get(parse_bat(sn), '0.6'), lw=0.8, alpha=0.35)
    agg = per.groupby('context_x')[col].agg(['mean', 'std', 'count'])
    ci = 1.96 * agg['std'] / np.sqrt(agg['count'])
    ax.errorbar(agg.index, agg['mean'], yerr=ci, fmt='o-', color='k', lw=2.5, capsize=3, zorder=5)
    ticks = CTX_WINDOWS + [13]
    ax.set(xlabel='context window (bins)', ylabel=f'session-median {lab}', ylim=ylim,
           title=f'ED 4{"a" if col == "fde" else "b"} — {lab}', xticks=ticks,
           xticklabels=[str(w) for w in CTX_WINDOWS] + ['full'])
ph.save_fig(fig_ed4, 'edfig4_context'); plt.show()
print('ED Fig 4a/b — encoding fit vs context window (session medians shown).')